# 12 — Interactive Dashboard Build and Data Lineage

## Germany’s Tourism Competitiveness in the European Union

**Purpose:** document and reproduce the data layer behind the interactive HTML dashboard.

This notebook is **connected to the outputs of the existing project notebooks**. It does not create a second, unrelated analysis. Instead, it uses the processed files created by notebooks 05–11 and converts those verified results into dashboard-ready data objects.

### Upstream analytical chain

- `05_germany_eu_benchmark.ipynb` → EU27 position and LOS counterfactual
- `06_source_market_analysis.ipynb` → source-market strategy
- `07_economic_analysis.ipynb` → travel receipts, expenditure and balance
- `08_2026_ytd_analysis.ipynb` → H1 2026 performance
- `09_machine_learning.ipynb` → seasonal-naive 2026 baseline
- `10_strategic_analysis.ipynb` → strategic recommendations
- `11_visualization_preparation.ipynb` → final Tableau/presentation export tables

The dashboard is therefore the **presentation layer of the analytical pipeline**, not a separate analysis.


## What to say if the panel asks, “Where did the dashboard data come from?”

> I did not manually type the analytical results into a separate presentation workflow. The dashboard is connected to the outputs of my project notebooks. The EU benchmark comes from Notebook 05, source-market analysis from Notebook 06, economic context from Notebook 07, 2026 YTD calculations from Notebook 08, the seasonal baseline from Notebook 09, and the recommendations from Notebook 10. Notebook 11 converts these results into presentation-ready export tables. This notebook loads those same outputs, validates the dashboard KPIs, and prepares the data objects used by the interactive HTML interface.

If they ask where the interface itself was made:

> The analytical work was done in Python/Jupyter and SQL, with Tableau used for visual validation. I then built the final presentation interface as a standalone HTML/CSS/JavaScript dashboard. The front-end is the visualization layer; the analytical values come from the notebook pipeline.


In [ ]:
# 1. Imports and project paths
# This follows the same project-root convention used in Notebooks 05, 08, 10 and 11.

import json
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)

project_root = Path.cwd().parent

processed_dir = (
    project_root
    / "data"
    / "processed"
)

tableau_dir = (
    project_root
    / "tableau"
)

presentation_dir = (
    project_root
    / "presentation"
)

presentation_dir.mkdir(
    parents=True,
    exist_ok=True
)

print("Project root:", project_root)
print("Processed data:", processed_dir)
print("Tableau exports:", tableau_dir)
print("Presentation:", presentation_dir)


## 2. Analytical lineage used by the dashboard

The code below explicitly maps every dashboard module to the notebook output that feeds it.


In [ ]:
dashboard_lineage = pd.DataFrame([
    {
        "dashboard_view": "KPI Overview",
        "upstream_notebook": "05_germany_eu_benchmark.ipynb + 08_2026_ytd_analysis.ipynb",
        "source_file": "germany_eu_los_benchmark_2021_2025.csv + germany_2026_ytd_summary.csv"
    },
    {
        "dashboard_view": "Competitive Analysis",
        "upstream_notebook": "05_germany_eu_benchmark.ipynb",
        "source_file": "eu27_tourism_benchmark_2024.csv + germany_los_counterfactual_scenarios_2024.csv"
    },
    {
        "dashboard_view": "Growth and Strategy",
        "upstream_notebook": "06_source_market_analysis.ipynb",
        "source_file": "germany_major_source_market_strategy_2024.csv"
    },
    {
        "dashboard_view": "Germany Tourism 2026",
        "upstream_notebook": "08_2026_ytd_analysis.ipynb + 09_machine_learning.ipynb",
        "source_file": "germany_2026_ytd_monthly_comparison.csv + germany_2026_actual_projection_combined.csv"
    },
    {
        "dashboard_view": "Travel Balance Context",
        "upstream_notebook": "07_economic_analysis.ipynb",
        "source_file": "germany_travel_bop_summary_2021_2025.csv"
    },
    {
        "dashboard_view": "Strategic Recommendations",
        "upstream_notebook": "10_strategic_analysis.ipynb",
        "source_file": "germany_strategic_recommendations.csv"
    },
    {
        "dashboard_view": "Final presentation exports",
        "upstream_notebook": "11_visualization_preparation.ipynb",
        "source_file": "tableau_*.csv"
    }
])

dashboard_lineage


## 3. Load the exact processed outputs

This is the key connection to the earlier notebooks.  
The dashboard notebook **reads their saved outputs** instead of recalculating everything independently.


In [ ]:
# Files generated by upstream notebooks

files = {
    "annual_los": processed_dir / "germany_eu_los_benchmark_2021_2025.csv",
    "eu_benchmark": processed_dir / "eu27_tourism_benchmark_2024.csv",
    "los_scenarios": processed_dir / "germany_los_counterfactual_scenarios_2024.csv",
    "source_markets": processed_dir / "germany_major_source_market_strategy_2024.csv",
    "travel_bop": processed_dir / "germany_travel_bop_summary_2021_2025.csv",
    "ytd_summary": processed_dir / "germany_2026_ytd_summary.csv",
    "ytd_monthly": processed_dir / "germany_2026_ytd_monthly_comparison.csv",
    "ml_combined": processed_dir / "germany_2026_actual_projection_combined.csv",
    "ml_summary": processed_dir / "germany_2026_baseline_scenario_summary.csv",
    "recommendations": processed_dir / "germany_strategic_recommendations.csv",
}

missing = [
    str(path)
    for path in files.values()
    if not path.exists()
]

if missing:
    print("Missing upstream outputs:")
    for path in missing:
        print(" -", path)
    print("\nRun the corresponding upstream notebooks first.")
else:
    print("All required processed outputs are available.")


In [ ]:
annual_los = pd.read_csv(files["annual_los"])
eu_benchmark = pd.read_csv(files["eu_benchmark"])
los_scenarios = pd.read_csv(files["los_scenarios"])
source_markets = pd.read_csv(files["source_markets"])
travel_bop = pd.read_csv(files["travel_bop"])
ytd_summary = pd.read_csv(files["ytd_summary"])
ytd_monthly = pd.read_csv(files["ytd_monthly"])
ml_combined = pd.read_csv(files["ml_combined"])
ml_summary = pd.read_csv(files["ml_summary"])
recommendations = pd.read_csv(files["recommendations"])

print("Annual LOS:", annual_los.shape)
print("EU benchmark:", eu_benchmark.shape)
print("LOS scenarios:", los_scenarios.shape)
print("Source markets:", source_markets.shape)
print("Travel BOP:", travel_bop.shape)
print("2026 YTD:", ytd_summary.shape)
print("2026 monthly:", ytd_monthly.shape)
print("2026 outlook:", ml_combined.shape)
print("Recommendations:", recommendations.shape)


## 4. Connection to Notebook 05 — Germany vs EU27

Notebook 05 calculates average length of stay and ranking from arrivals and nights.

The original project logic is:

```python
benchmark_2024["avg_length_of_stay"] = (
    benchmark_2024["foreign_nights"]
    / benchmark_2024["foreign_arrivals"]
)

benchmark_2024["los_rank"] = (
    benchmark_2024["avg_length_of_stay"]
    .rank(method="min", ascending=False)
    .astype(int)
)
```

The dashboard uses the **saved result of that code**, not a different formula.


In [ ]:
# Dashboard extract: Germany's verified 2024 EU position

germany_2024 = (
    eu_benchmark
    .loc[eu_benchmark["country_code"] == "DE"]
    .iloc[0]
)

competitive_kpis = {
    "foreign_arrivals": float(germany_2024["foreign_arrivals"]),
    "arrivals_rank": int(germany_2024["arrivals_rank"]),
    "foreign_nights": float(germany_2024["foreign_nights"]),
    "nights_rank": int(germany_2024["nights_rank"]),
    "avg_length_of_stay": float(germany_2024["avg_length_of_stay"]),
    "los_rank": int(germany_2024["los_rank"]),
}

competitive_kpis


In [ ]:
# Counterfactual scenarios shown in Competitive Analysis

counterfactual_dashboard = (
    los_scenarios[
        [
            "benchmark",
            "benchmark_los",
            "illustrative_nights",
            "additional_nights_vs_actual",
            "additional_nights_pct"
        ]
    ]
    .copy()
)

counterfactual_dashboard.round(2)


### If asked: “Did you forecast the additional 8.4 million nights?”

> No. Notebook 05 defines this as a counterfactual scenario. I hold Germany’s 2024 foreign arrivals constant and apply a benchmark length of stay. It illustrates the magnitude of the stay-duration gap. It is not a forecast.


## 5. Connection to Notebook 08 — January–June 2026

Notebook 08 calculates year-on-year changes with `pct_change()`:

```python
germany_h1_summary["arrivals_growth_pct"] = (
    germany_h1_summary["foreign_arrivals"].pct_change() * 100
)

germany_h1_summary["nights_growth_pct"] = (
    germany_h1_summary["foreign_nights"].pct_change() * 100
)

germany_h1_summary["los_change_pct"] = (
    germany_h1_summary["avg_length_of_stay"].pct_change() * 100
)
```

This notebook reads the output `germany_2026_ytd_summary.csv` created by Notebook 08.


In [ ]:
# Keep only the 2026 row for the dashboard KPI layer

ytd_2026 = (
    ytd_summary
    .loc[ytd_summary["year"] == 2026]
    .iloc[0]
)

ytd_kpis = {
    "foreign_arrivals": float(ytd_2026["foreign_arrivals"]),
    "foreign_nights": float(ytd_2026["foreign_nights"]),
    "avg_length_of_stay": float(ytd_2026["avg_length_of_stay"]),
    "arrivals_growth_pct": float(ytd_2026["arrivals_growth_pct"]),
    "nights_growth_pct": float(ytd_2026["nights_growth_pct"]),
    "los_change_pct": float(ytd_2026["los_change_pct"]),
}

ytd_kpis


In [ ]:
# Monthly 2026 chart data

monthly_2026_dashboard = (
    ytd_monthly[
        [
            "month",
            "arrivals_growth_pct",
            "nights_growth_pct",
            "los_change_pct"
        ]
    ]
    .copy()
)

monthly_2026_dashboard


## 6. Connection to Notebook 09 — 2026 seasonal baseline

Notebook 09 compares predictive approaches using chronological and rolling out-of-sample validation.  
The final project retained the **seasonal-naive model** because it performed best on the rolling evaluation.

The dashboard therefore reads the actual/projection file created by Notebook 09 rather than manually entering projected monthly values.


In [ ]:
# 2026 observed + seasonal-naive projection

outlook_2026_dashboard = ml_combined.copy()

# Convert date if present
if "date" in outlook_2026_dashboard.columns:
    outlook_2026_dashboard["date"] = pd.to_datetime(
        outlook_2026_dashboard["date"]
    )

outlook_2026_dashboard


In [ ]:
# Read the model summary used for the strategic interpretation

ml_summary


### If asked: “Why use a simple seasonal-naive model?”

> I selected the model based on out-of-sample performance, not model complexity. Tourism is strongly seasonal, and the seasonal-naive benchmark performed better than the tested regression and random-forest alternatives in the rolling evaluation. A more complex model is not automatically a better model.


## 7. Connection to Notebook 06 — Source-market strategy

The source-market dashboard uses the processed output created by Notebook 06.

It keeps the project dimensions already defined there:
- foreign overnight scale
- 2023–2024 growth
- average length of stay
- strategic role
- stay-extension priority


In [ ]:
source_market_dashboard = (
    source_markets[
        [
            "source_market_code",
            "source_market",
            "foreign_nights",
            "growth_2023_2024_pct",
            "avg_length_of_stay",
            "strategic_role",
            "stay_extension_priority"
        ]
    ]
    .copy()
)

source_market_dashboard.head(20)


## 8. Connection to Notebook 07 — Travel Balance Context

The dashboard does **not** derive economic values from accommodation statistics.

It reads the Balance of Payments travel summary produced by Notebook 07, keeping the statistical systems separate.


In [ ]:
economic_dashboard = (
    travel_bop[
        [
            "year",
            "travel_receipts_mio_eur",
            "travel_expenditure_mio_eur",
            "travel_balance_mio_eur",
            "receipts_cover_expenditure_pct",
            "is_provisional"
        ]
    ]
    .copy()
)

economic_dashboard


## 9. Connection to Notebook 10 — Strategic Recommendations

Notebook 10 integrates the benchmark, source-market, YTD, economic and predictive evidence.

The current dashboard intentionally displays only:
1. Strategic Priority
2. Recommended Direction
3. Evidence Basis

The **Selected Year Context has been removed** from the recommendation panel.


In [ ]:
recommendation_dashboard = (
    recommendations[
        [
            "priority",
            "strategic_priority",
            "recommended_direction",
            "evidence_basis",
            "primary_business_kpi"
        ]
    ]
    .sort_values("priority")
    .reset_index(drop=True)
)

recommendation_dashboard


## 10. Connection to Notebook 11 — presentation-ready exports

Notebook 11 creates the final visualization tables used for Tableau and the HTML presentation layer.

The project code exports:

- `tableau_eu_competitive_position.csv`
- `tableau_eu_los_2024.csv`
- `tableau_los_scenarios_2024.csv`
- `tableau_source_market_strategy_2024.csv`
- `tableau_2026_ytd_growth.csv`
- `tableau_2026_outlook.csv`
- `tableau_economic_context.csv`
- `tableau_strategic_recommendations.csv`

This is the cleanest point of connection between the analytical notebooks and the presentation dashboard.


In [ ]:
tableau_files = {
    "competitive": tableau_dir / "tableau_eu_competitive_position.csv",
    "eu_los": tableau_dir / "tableau_eu_los_2024.csv",
    "counterfactual": tableau_dir / "tableau_los_scenarios_2024.csv",
    "source_market": tableau_dir / "tableau_source_market_strategy_2024.csv",
    "ytd_growth": tableau_dir / "tableau_2026_ytd_growth.csv",
    "outlook": tableau_dir / "tableau_2026_outlook.csv",
    "economic": tableau_dir / "tableau_economic_context.csv",
    "recommendations": tableau_dir / "tableau_strategic_recommendations.csv",
}

for name, path in tableau_files.items():
    print(f"{name:16s}", "OK" if path.exists() else "MISSING", path)


In [ ]:
# Load Notebook 11's presentation-ready tables when available

tableau_data = {}

for name, path in tableau_files.items():
    if path.exists():
        tableau_data[name] = pd.read_csv(path)

for name, df in tableau_data.items():
    print(name, df.shape)


## 11. Dashboard validation

Before building the HTML payload, validate the headline values against the analytical outputs.  
These are **quality-control checks**, not manually entered dashboard assumptions.


In [ ]:
# Use tolerances because source files may store full precision.

assert competitive_kpis["arrivals_rank"] == 4
assert competitive_kpis["nights_rank"] == 7
assert competitive_kpis["los_rank"] == 21
assert np.isclose(competitive_kpis["avg_length_of_stay"], 2.27, atol=0.02)

assert np.isclose(ytd_kpis["arrivals_growth_pct"], 2.04, atol=0.05)
assert np.isclose(ytd_kpis["nights_growth_pct"], 0.74, atol=0.05)
assert np.isclose(ytd_kpis["los_change_pct"], -1.27, atol=0.05)

print("Dashboard KPI validation passed.")


## 12. Convert notebook outputs into JavaScript-ready objects

This is the bridge between Python/Jupyter and the HTML dashboard.

Python performs the analysis.  
JavaScript receives the verified output as JSON and controls the interactive presentation.


In [ ]:
def records(df):
    """Convert a DataFrame into browser-safe records."""
    clean = df.copy()

    for col in clean.columns:
        if pd.api.types.is_datetime64_any_dtype(clean[col]):
            clean[col] = clean[col].dt.strftime("%Y-%m-%d")

    clean = clean.replace({np.nan: None})

    return clean.to_dict(orient="records")


dashboard_data = {
    "competitive_kpis": competitive_kpis,
    "eu_competitive_position": records(eu_benchmark),
    "los_counterfactual": records(counterfactual_dashboard),
    "source_markets": records(source_market_dashboard),
    "ytd_kpis": ytd_kpis,
    "ytd_monthly": records(monthly_2026_dashboard),
    "outlook_2026": records(outlook_2026_dashboard),
    "economic_context": records(economic_dashboard),
    "strategic_recommendations": records(recommendation_dashboard),
}

print(dashboard_data.keys())


In [ ]:
# Write the exact analytical payload used by the front-end

dashboard_json_path = (
    presentation_dir
    / "germany_tourism_dashboard_data.json"
)

dashboard_json_path.write_text(
    json.dumps(
        dashboard_data,
        indent=2,
        ensure_ascii=False
    ),
    encoding="utf-8"
)

print("Saved:", dashboard_json_path)


## 13. How the HTML dashboard uses this data

A standalone browser dashboard cannot directly execute a Jupyter notebook.  
The correct architecture is:

**Notebook analysis → processed CSVs → dashboard JSON/JavaScript → HTML/CSS/SVG**

For example, JavaScript can receive the Python-generated object like this:

```javascript
const germany2024 = DASHBOARD_DATA.competitive_kpis;

document.querySelector("#arrivalsRank").textContent =
    "#" + germany2024.arrivals_rank;
```

And a chart can be generated from:

```javascript
const monthly = DASHBOARD_DATA.ytd_monthly;

monthly.forEach(row => {
    console.log(
        row.month,
        row.arrivals_growth_pct,
        row.nights_growth_pct,
        row.los_change_pct
    );
});
```

This separation is deliberate:
- Python handles analysis and validation.
- HTML/CSS handles layout.
- JavaScript handles interaction.
- SVG renders the charts in the browser.


## 14. Export a JavaScript data file

This allows the front-end code to use the exact notebook-generated payload without manually retyping values.


In [ ]:
dashboard_js_path = (
    presentation_dir
    / "germany_tourism_dashboard_data.js"
)

dashboard_js = (
    "window.DASHBOARD_DATA = "
    + json.dumps(
        dashboard_data,
        ensure_ascii=False
    )
    + ";"
)

dashboard_js_path.write_text(
    dashboard_js,
    encoding="utf-8"
)

print("Saved:", dashboard_js_path)


## 15. Current HTML dashboard source

The next cell embeds the current V36.3 dashboard source so this notebook documents both sides of the implementation:

1. the **analytical data lineage**, and
2. the **front-end HTML/CSS/JavaScript implementation**.

The cell is collapsed by default because the HTML source is long.


In [ ]:
DASHBOARD_HTML_SOURCE = r"""<!DOCTYPE html>
<html lang="en">
<head>
<meta charset="utf-8">
<meta name="viewport" content="width=device-width, initial-scale=1">
<title>Germany Tourism Intelligence Dashboard</title>
<style>
:root{
  --nav:#08131e; --nav2:#0e1d2b; --paper:#f4f7fa; --card:#ffffff;
  --text:#101820; --muted:#667484; --red:#d71920; --gold:#f1b514;
  --green:#159a63; --blue:#1565c0; --purple:#7047a8; --line:#dfe6ec;
}
*{box-sizing:border-box}
body{margin:0;background:#0a1119;font-family:Inter,Arial,sans-serif;color:var(--text)}
.app{width:min(1600px,100%);margin:auto;min-height:900px;background:var(--paper);display:grid;grid-template-columns:310px 1fr;overflow:hidden}
.sidebar{background:linear-gradient(180deg,var(--nav),#071018);color:#fff;padding:20px 16px;display:flex;flex-direction:column;gap:14px}
.brand{display:flex;gap:12px;align-items:center;padding:4px 4px 12px}
.flag{width:54px;height:36px;display:grid;overflow:hidden;border-radius:2px;box-shadow:0 0 0 1px #ffffff22}
.flag i:nth-child(1){background:#111}.flag i:nth-child(2){background:#cf202f}.flag i:nth-child(3){background:#f2c230}
.brand h1{font-size:24px;line-height:1;margin:0;letter-spacing:.04em}
.brand small{display:block;color:#ccd5de;margin-top:5px;letter-spacing:.06em}
.side-label{font-size:12px;text-transform:uppercase;letter-spacing:.12em;color:#9eacb9;margin:5px 8px 0}
.qbtn{width:100%;min-height:66px;border-radius:14px;border:1px solid #26384a;background:#101d29;color:#fff;display:grid;grid-template-columns:38px 1fr;align-items:center;text-align:left;padding:10px 14px;cursor:pointer;font-size:15px;font-weight:650;transition:.2s}
.qbtn:hover{background:#162737;border-color:#476079}
.qbtn.active{background:linear-gradient(90deg,#786015,#c49b1b);box-shadow:0 0 0 2px #ffd75a,0 0 20px #f0b40055}
.qnum{font-size:19px;color:#dce4eb}
.side-photo{margin-top:auto;min-height:215px;border-radius:14px;background-size:cover;background-position:center;position:relative;overflow:hidden;border:1px solid #26384a}
.side-photo:after{content:"";position:absolute;inset:0;background:linear-gradient(transparent 35%,rgba(0,0,0,.85))}
.side-photo .caption{position:absolute;z-index:2;bottom:16px;left:16px;right:16px}
.side-photo .caption b{font-size:28px;display:block}.side-photo .caption span{font-size:13px;color:#e3e8ec}
.main{min-width:0;background:var(--paper)}
.hero{min-height:176px;background-size:cover;background-position:center;position:relative;color:#fff;padding:22px 26px;display:flex;align-items:flex-end}
.hero:after{content:"";position:absolute;inset:0;background:linear-gradient(90deg,rgba(4,10,16,.84),rgba(4,10,16,.22),rgba(4,10,16,.45))}
.hero-content{position:relative;z-index:2;width:100%;display:flex;align-items:end;justify-content:space-between;gap:20px}
.eyebrow{font-size:13px;letter-spacing:.08em;text-transform:uppercase;color:#dde5eb}
.hero h2{font-size:34px;line-height:1.05;margin:7px 0 0}
.hero .sub{font-size:14px;margin-top:7px;color:#e5eaee}
.period{background:#fff;color:#111;border:0;border-radius:24px;padding:11px 16px;font-weight:700}
.content{padding:16px 18px 20px}
.kpis{display:grid;grid-template-columns:repeat(4,1fr);gap:12px;margin-bottom:12px}
.kpi{background:var(--card);border:1px solid var(--line);border-radius:13px;padding:15px 16px;cursor:pointer;min-height:104px;transition:.18s}
.kpi:hover{transform:translateY(-1px);border-color:#b9c5ce}
.kpi .label{font-size:13px;color:#465563;font-weight:700}.kpi .value{font-size:29px;font-weight:800;margin-top:7px}
.kpi .note{font-size:12px;color:var(--muted);margin-top:4px}.positive{color:var(--green)!important}.negative{color:var(--red)!important}
.grid{display:grid;grid-template-columns:minmax(0,1.6fr) minmax(290px,.65fr);gap:12px}
.card{background:#fff;border:1px solid var(--line);border-radius:13px;overflow:hidden}
.card-head{display:flex;align-items:start;justify-content:space-between;gap:12px;padding:16px 18px 5px}
.card h3{margin:0;font-size:20px}.card .desc{font-size:13px;color:var(--muted);margin-top:4px}
.chart-wrap{padding:5px 16px 15px}
svg{width:100%;height:auto;display:block}
.photo-card{min-height:355px;background-size:cover;background-position:center;position:relative;color:#fff;cursor:pointer}
.photo-card:after{content:"";position:absolute;inset:0;background:linear-gradient(transparent 35%,rgba(0,0,0,.86))}
.photo-copy{position:absolute;z-index:2;bottom:0;left:0;right:0;padding:18px}
.photo-copy h3{font-size:24px}.photo-copy p{margin:6px 0 0;font-size:13px;color:#e8edf1}
.slideshow-controls{position:absolute;z-index:5;top:12px;right:12px;display:flex;gap:6px}
.slide-btn{width:38px;height:38px;border-radius:50%;border:1px solid #ffffff66;background:#071018bb;color:#fff;font-size:16px;cursor:pointer;display:grid;place-items:center;backdrop-filter:blur(4px)}
.slide-btn:hover{background:#071018ee}
.slide-counter{position:absolute;z-index:5;top:16px;left:16px;background:#071018bb;color:#fff;border:1px solid #ffffff44;border-radius:18px;padding:7px 10px;font-size:11px;letter-spacing:.04em;backdrop-filter:blur(4px)}
.photo-card,.hero,.side-photo{transition:background-image .55s ease-in-out,opacity .25s ease}
.bottom{display:grid;grid-template-columns:minmax(0,1fr) minmax(330px,.8fr);gap:12px;margin-top:12px}
.insight{padding:17px 18px}
.insight-grid{display:grid;grid-template-columns:1fr 1fr;gap:14px}
.insight h4{font-size:12px;text-transform:uppercase;letter-spacing:.08em;color:var(--muted);margin:0 0 7px}
.insight p{margin:0;line-height:1.46;font-size:14px}
.big-number{font-size:42px;font-weight:900;color:var(--red)}
.flash{animation:flash .7s ease}
@keyframes flash{0%{box-shadow:0 0 0 0 #f1b51400}42%{box-shadow:0 0 0 7px #f1b51455;transform:scale(1.008)}100%{box-shadow:none;transform:scale(1)}}
.dim{opacity:.28;filter:saturate(.6);transition:.2s}
.tooltip{white-space:pre-line;position:absolute;pointer-events:none;background:#08131e;color:#fff;padding:8px 10px;border-radius:8px;font-size:12px;box-shadow:0 8px 24px #0004;display:none;z-index:20;max-width:220px}
.source{font-size:10px;color:#788694;margin-top:12px}
.legend{display:flex;gap:16px;flex-wrap:wrap;font-size:12px;color:#576675;padding:0 18px 14px}
.dot{width:9px;height:9px;border-radius:50%;display:inline-block;margin-right:5px}
@media(max-width:1100px){.app{grid-template-columns:240px 1fr}.kpis{grid-template-columns:repeat(2,1fr)}.grid,.bottom{grid-template-columns:1fr}.photo-card{min-height:250px}}
@media(max-width:760px){.app{display:block}.sidebar{padding:12px}.side-photo{display:none}.qbtn{min-height:50px}.hero{min-height:145px}.kpis{grid-template-columns:1fr 1fr}.hero h2{font-size:26px}}

.analysis-menu{margin:18px 0 12px}
.analysis-menu label{display:block;font-size:10px;font-weight:800;letter-spacing:.12em;text-transform:uppercase;color:#a8b2bc;margin:0 0 7px 2px}
.analysis-menu select{width:100%;background:#101f2d;color:#fff;border:1px solid #ffffff24;border-radius:10px;padding:12px 34px 12px 12px;font-size:13px;font-weight:700;outline:none;cursor:pointer}
.analysis-menu select:hover,.analysis-menu select:focus{border-color:#d71920;box-shadow:0 0 0 2px #d7192020}

.period-control{display:flex;align-items:center;gap:8px}
.period-control label{font-size:11px;font-weight:800;letter-spacing:.08em;text-transform:uppercase;color:#5c6873}
.period-control.hidden{display:none}


.evidence-card{margin-top:12px;background:#fff;border:1px solid #dfe5ea;border-radius:14px;padding:14px 16px;box-shadow:0 8px 22px rgba(8,19,30,.05)}
.evidence-card h3{margin:0 0 10px;font-size:14px}
.evidence-grid{display:grid;grid-template-columns:repeat(3,minmax(0,1fr));gap:10px}
.evidence-item{background:#f6f8fa;border-radius:10px;padding:10px 12px;min-height:72px}
.evidence-item .e-title{font-size:10px;font-weight:800;letter-spacing:.05em;text-transform:uppercase;color:#6e7983}
.evidence-item .e-value{font-size:18px;font-weight:800;margin-top:4px}
.evidence-item .e-note{font-size:11px;color:#59636d;margin-top:3px;line-height:1.35}
@media(max-width:900px){.evidence-grid{grid-template-columns:1fr}}


/* V11: Move visual emphasis to left photo rail and widen analytical workspace */
.app{
  grid-template-columns:260px minmax(0,1fr) !important;
}
.sidebar{
  width:auto !important;
}
.side-photo{
  min-height:420px !important;
  height:calc(100vh - 290px) !important;
  background-size:cover !important;
  background-position:center !important;
}
.grid{
  grid-template-columns:minmax(0,1fr) !important;
}
.photo-card{
  display:none !important;
}
#graphCard{
  grid-column:1 / -1 !important;
}
.chart-wrap{
  min-height:360px !important;
}
.chart-wrap svg{
  width:100% !important;
  max-width:none !important;
}
@media(max-width:1100px){
  .app{grid-template-columns:220px minmax(0,1fr) !important}
}
@media(max-width:760px){
  .app{display:block !important}
  .side-photo{display:none !important}
}


.strategy-priority{transition:opacity .25s ease,transform .25s ease,box-shadow .25s ease,border-color .25s ease}
.strategy-priority.focused{transform:scale(1.015);box-shadow:0 0 0 3px rgba(215,25,32,.18),0 10px 28px rgba(8,19,30,.10);border-color:#d71920 !important}
.strategy-priority.faded{opacity:.22}
.strategy-action,.strategy-evidence{transition:background .25s ease,box-shadow .25s ease,border-color .25s ease}
.strategy-priority.focused .strategy-action{background:#fff7f7 !important;border-color:#d71920 !important;box-shadow:0 0 0 2px rgba(215,25,32,.10)}
.strategy-evidence{display:none;margin-top:9px;padding:10px 12px;border-radius:9px;background:#fffdf4;border:1px solid #e6c861;color:#33414d;font-size:12px;line-height:1.45}
.strategy-priority.focused .strategy-evidence{display:block;animation:strategyPulse .55s ease}
@keyframes strategyPulse{
 0%{transform:scale(.985);box-shadow:0 0 0 0 rgba(241,181,20,.0)}
 45%{transform:scale(1.01);box-shadow:0 0 0 5px rgba(241,181,20,.18)}
 100%{transform:scale(1);box-shadow:0 0 0 0 rgba(241,181,20,.0)}
}


.strategy-tabs{
  display:flex;
  gap:8px;
  margin:8px 0 16px;
  flex-wrap:wrap;
}
.strategy-tab{
  border:1px solid #cfd7de;
  background:#fff;
  color:#22313f;
  padding:9px 16px;
  border-radius:8px;
  font-size:13px;
  font-weight:800;
  cursor:pointer;
  transition:.2s ease;
}
.strategy-tab:hover{
  border-color:#d71920;
  color:#d71920;
}
.strategy-tab.active{
  background:#08131e;
  color:#fff;
  border-color:#08131e;
}
.strategy-detail{
  border:1px solid #dfe6ec;
  background:#f8fafb;
  border-radius:14px;
  padding:20px;
  min-height:250px;
}
.strategy-field{
  padding:14px 0;
  border-bottom:1px solid #e3e8ec;
}
.strategy-field:last-child{
  border-bottom:0;
}
.strategy-field-label{
  font-size:10px;
  text-transform:uppercase;
  letter-spacing:.08em;
  font-weight:800;
  color:#7a8793;
  margin-bottom:6px;
}
.strategy-field-value{
  font-size:16px;
  line-height:1.45;
  color:#22313f;
  font-weight:600;
}
.strategy-field-value.evidence{
  font-size:14px;
  font-weight:500;
  color:#44515d;
}


/* V15: strategic recommendation uses the existing large KPI tabs only */
.strategy-tabs{display:none !important}
.strategy-detail{margin-top:6px}
.strategy-kpi-active{
  border-color:#d71920 !important;
  box-shadow:0 0 0 2px rgba(215,25,32,.12) !important;
  transform:translateY(-1px);
}


/* V25 review: use the Tableau chart titles only inside analytical charts */
body[data-view="competition"] #graphCard .card-head,
body[data-view="markets"] #graphCard .card-head,
body[data-view="ytd"] #graphCard .card-head,
body[data-view="economy"] #graphCard .card-head{
  display:none;
}
body[data-view="competition"] #chartWrap,
body[data-view="markets"] #chartWrap,
body[data-view="ytd"] #chartWrap,
body[data-view="economy"] #chartWrap{
  padding-top:14px;
}
.side-photo{
  background-size:cover!important;
  background-position:center!important;
  image-rendering:auto;
}


/* V26 Closing: visual photo-slide only */
body[data-view="closing"] #kpis,
body[data-view="closing"] #evidenceCard,
body[data-view="closing"] #finding,
body[data-view="closing"] #implication,
body[data-view="closing"] #transition,
body[data-view="closing"] .finding,
body[data-view="closing"] .implication,
body[data-view="closing"] .transition,
body[data-view="closing"] .insight-grid,
body[data-view="closing"] #graphCard .card-head {
  display:none !important;
}
body[data-view="closing"] #graphCard{
  grid-column:1/-1;
  border:0;
  box-shadow:none;
  background:transparent;
}
body[data-view="closing"] #chartWrap{
  padding:0 !important;
}


/* V31 Closing cleanup */
body[data-view="closing"] #heroTitle,
body[data-view="closing"] #heroSub,
body[data-view="closing"] #transitionCard {
  display: none !important;
}


/* V32: MacBook Pro 14 / viewport-fit presentation mode */
html, body {
  width: 100%;
  height: 100%;
  margin: 0;
  overflow: hidden !important;
}

body {
  min-height: 100dvh;
}

#viewportStage {
  position: fixed;
  inset: 0;
  overflow: hidden;
  background: #0a1119;
}

.app {
  width: 100% !important;
  max-width: none !important;
  min-height: 0 !important;
  height: 100% !important;
  margin: 0 !important;
  transform-origin: top left;
}

/* Compact vertical rhythm for laptop presentation screens */
@media (min-width: 900px) {
  .sidebar {
    padding: 14px 12px !important;
    gap: 9px !important;
  }
  .brand {
    padding-bottom: 6px !important;
  }
  .brand h1 {
    font-size: 21px !important;
  }
  .analysis-menu {
    margin: 8px 0 7px !important;
  }
  .analysis-menu select {
    padding-top: 9px !important;
    padding-bottom: 9px !important;
  }
  .side-photo {
    min-height: 0 !important;
    height: auto !important;
    flex: 1 1 auto !important;
  }

  .hero {
    min-height: 122px !important;
    height: 122px !important;
    padding: 14px 20px !important;
  }
  .hero h2 {
    font-size: clamp(25px, 2.2vw, 31px) !important;
  }
  .hero .sub {
    margin-top: 4px !important;
    font-size: 12px !important;
  }
  .eyebrow {
    font-size: 11px !important;
  }

  .content {
    padding: 10px 14px 10px !important;
    height: calc(100% - 122px);
    overflow: hidden !important;
    display: flex;
    flex-direction: column;
  }

  .kpis {
    gap: 8px !important;
    margin-bottom: 8px !important;
    flex: 0 0 auto;
  }
  .kpi {
    min-height: 76px !important;
    padding: 9px 12px !important;
  }
  .kpi .label {
    font-size: 11px !important;
  }
  .kpi .value {
    font-size: clamp(21px, 2vw, 26px) !important;
    margin-top: 4px !important;
  }
  .kpi .note {
    font-size: 10px !important;
    margin-top: 2px !important;
  }

  .grid {
    gap: 8px !important;
    min-height: 0;
    flex: 1 1 auto;
  }
  #graphCard {
    min-height: 0 !important;
  }
  .chart-wrap {
    min-height: 0 !important;
    padding: 4px 12px 8px !important;
  }
  .chart-wrap svg {
    max-height: 43vh !important;
    object-fit: contain;
  }
  .card-head {
    padding: 9px 12px 3px !important;
  }
  .card h3 {
    font-size: 17px !important;
  }
  .card .desc {
    font-size: 11px !important;
  }
  .legend {
    padding: 0 12px 8px !important;
    font-size: 10px !important;
  }

  .bottom {
    gap: 8px !important;
    margin-top: 8px !important;
    flex: 0 0 auto;
  }
  .insight {
    padding: 10px 12px !important;
  }
  .insight h4 {
    margin-bottom: 4px !important;
    font-size: 10px !important;
  }
  .insight p {
    font-size: 12px !important;
    line-height: 1.32 !important;
  }

  .evidence-card {
    margin-top: 8px !important;
    padding: 8px 10px !important;
    flex: 0 0 auto;
  }
  .evidence-card h3 {
    margin-bottom: 6px !important;
    font-size: 12px !important;
  }
  .evidence-item {
    min-height: 52px !important;
    padding: 7px 9px !important;
  }
  .evidence-item .e-value {
    font-size: 15px !important;
  }
  .evidence-item .e-note {
    font-size: 9px !important;
  }
  .source {
    margin-top: 6px !important;
    font-size: 9px !important;
    flex: 0 0 auto;
  }

  .strategy-detail {
    min-height: 0 !important;
    padding: 12px !important;
  }
  .strategy-field {
    padding: 8px 0 !important;
  }
  .strategy-field-value {
    font-size: 13px !important;
    line-height: 1.32 !important;
  }
  .strategy-field-value.evidence {
    font-size: 12px !important;
  }

  body[data-view="closing"] .hero {
    height: 62px !important;
    min-height: 62px !important;
  }
  body[data-view="closing"] .content {
    height: calc(100% - 62px) !important;
    padding-top: 8px !important;
  }
  body[data-view="closing"] .closing-photo-stage {
    min-height: 0 !important;
    height: calc(100dvh - 86px) !important;
    max-height: none !important;
  }
}

/* User-controlled fit toolbar */
.fit-toolbar {
  position: fixed;
  right: 12px;
  bottom: 12px;
  z-index: 9999;
  display: flex;
  align-items: center;
  gap: 5px;
  padding: 5px;
  border-radius: 22px;
  background: rgba(8,19,30,.88);
  box-shadow: 0 6px 20px rgba(0,0,0,.25);
  backdrop-filter: blur(8px);
}
.fit-toolbar button {
  min-width: 34px;
  height: 30px;
  border: 1px solid rgba(255,255,255,.18);
  border-radius: 16px;
  background: rgba(255,255,255,.08);
  color: #fff;
  font: 700 12px Inter,Arial,sans-serif;
  cursor: pointer;
}
.fit-toolbar button:hover {
  background: rgba(255,255,255,.16);
}
.fit-toolbar .fit-label {
  min-width: 48px;
  color: #fff;
  text-align: center;
  font-size: 10px;
  font-weight: 800;
  letter-spacing: .02em;
}


/* V33: true whole-dashboard fit, nothing clipped */
html, body {
  overflow: hidden !important;
}

#viewportStage {
  position: fixed;
  inset: 0;
  overflow: hidden !important;
}

.app {
  width: 1440px !important;
  max-width: none !important;
  height: auto !important;
  min-height: 900px !important;
  margin: 0 !important;
  transform-origin: top left !important;
}

/* Do not clip internal dashboard content. The whole page is scaled instead. */
.main,
.content,
.grid,
#graphCard,
.chart-wrap {
  height: auto !important;
  max-height: none !important;
  overflow: visible !important;
}

.content {
  display: block !important;
}

.chart-wrap {
  min-height: 0 !important;
}

.chart-wrap svg {
  max-height: none !important;
  width: 100% !important;
  height: auto !important;
}

.side-photo {
  min-height: 360px !important;
  height: auto !important;
  flex: 1 1 auto !important;
}

/* Closing uses its natural full content height and is then scaled as one page. */
body[data-view="closing"] .closing-photo-stage {
  height: 650px !important;
  min-height: 650px !important;
  max-height: none !important;
}

/* Preserve readable spacing at the natural design size. */
.hero {
  height: auto !important;
  min-height: 150px !important;
}

body[data-view="closing"] .hero {
  min-height: 70px !important;
}

@media (max-width: 900px) {
  .app {
    width: 1180px !important;
  }
}


/* V34: extra-safe fit for the two stacked-chart views */
body[data-view="competition"] .hero,
body[data-view="ytd"] .hero {
  min-height: 112px !important;
  padding-top: 10px !important;
  padding-bottom: 10px !important;
}

body[data-view="competition"] .content,
body[data-view="ytd"] .content {
  padding-top: 7px !important;
  padding-bottom: 7px !important;
}

body[data-view="competition"] .kpis,
body[data-view="ytd"] .kpis {
  margin-bottom: 6px !important;
}

body[data-view="competition"] .kpi,
body[data-view="ytd"] .kpi {
  min-height: 70px !important;
  padding: 7px 10px !important;
}

body[data-view="competition"] .bottom,
body[data-view="ytd"] .bottom {
  margin-top: 6px !important;
}

body[data-view="competition"] .insight,
body[data-view="ytd"] .insight {
  padding: 8px 10px !important;
}

body[data-view="competition"] .source,
body[data-view="ytd"] .source {
  margin-top: 4px !important;
}


/* V35: MacBook Pro 14 fit by reflow, not by shrinking */
.dual-chart-grid{
  display:grid;
  grid-template-columns:repeat(2,minmax(0,1fr));
  gap:12px;
  width:100%;
}
.dual-chart-panel{
  min-width:0;
  background:#fff;
  border:1px solid #e4e9ed;
  border-radius:10px;
  padding:10px 10px 6px;
}
.dual-chart-title{
  font-size:15px;
  font-weight:800;
  line-height:1.15;
  margin:0 0 3px;
}
.dual-chart-subtitle{
  font-size:10.5px;
  color:#687580;
  line-height:1.25;
  min-height:26px;
  margin-bottom:3px;
}
.dual-chart-panel svg{
  width:100%;
  height:auto;
  max-height:310px;
}

/* Fit the MacBook Pro 14 browser viewport primarily through layout */
@media (min-width:1200px) and (max-height:950px){
  .app{
    width:100% !important;
    min-height:100vh !important;
    grid-template-columns:220px minmax(0,1fr) !important;
  }
  .sidebar{
    padding:11px 10px !important;
  }
  .brand h1{font-size:19px !important}
  .brand small{font-size:10px !important}
  .analysis-menu{margin:6px 0 !important}
  .side-photo{min-height:250px !important}

  .hero{
    min-height:96px !important;
    padding:10px 16px !important;
  }
  .hero h2{
    font-size:25px !important;
  }
  .hero .sub{
    font-size:11px !important;
  }

  .content{
    padding:7px 10px 8px !important;
  }

  .kpis{
    gap:7px !important;
    margin-bottom:7px !important;
  }
  .kpi{
    min-height:62px !important;
    padding:7px 9px !important;
  }
  .kpi .label{font-size:10px !important}
  .kpi .value{font-size:21px !important}
  .kpi .note{font-size:9px !important}

  .chart-wrap{
    padding:5px 8px 5px !important;
  }

  .bottom{
    margin-top:6px !important;
    gap:7px !important;
  }
  .insight{
    padding:7px 9px !important;
  }
  .insight h4{font-size:9px !important;margin-bottom:3px !important}
  .insight p{font-size:10.5px !important;line-height:1.25 !important}
  .source{font-size:8.5px !important;margin-top:4px !important}
  .legend{font-size:9px !important;padding:0 10px 5px !important}

  body[data-view="competition"] .dual-chart-panel svg,
  body[data-view="ytd"] .dual-chart-panel svg{
    max-height:285px;
  }
}

/* Narrower windows re-stack, and the normal adaptive fit takes over */
@media (max-width:1050px){
  .dual-chart-grid{grid-template-columns:1fr}
  .dual-chart-panel svg{max-height:none}
}


/* V36: use available MacBook Pro 14 space and improve presentation readability */
.dual-chart-grid{
  align-items:stretch;
}
.dual-chart-panel{
  display:flex;
  flex-direction:column;
  justify-content:flex-start;
}
.dual-chart-panel svg{
  flex:1 1 auto;
}

/* Strategic Recommendations should use the available canvas instead of leaving an empty block */
body[data-view="strategy"] #chartWrap{
  padding:8px 12px 10px !important;
}
body[data-view="strategy"] .strategy-detail{
  min-height:330px !important;
  display:flex;
  flex-direction:column;
  justify-content:center;
  padding:22px 26px !important;
}
body[data-view="strategy"] .strategy-field{
  padding:13px 0 !important;
}
body[data-view="strategy"] .strategy-field-label{
  font-size:11px !important;
}
body[data-view="strategy"] .strategy-field-value{
  font-size:16px !important;
  line-height:1.42 !important;
}
body[data-view="strategy"] .strategy-field-value.evidence{
  font-size:14px !important;
  line-height:1.45 !important;
}

/* Bottom interpretation text must remain presentation-readable */
.bottom .insight h4{
  font-size:11px !important;
  line-height:1.2 !important;
}
.bottom .insight p{
  font-size:13.5px !important;
  line-height:1.38 !important;
}
.source{
  font-size:10.5px !important;
  line-height:1.3 !important;
}
.legend{
  font-size:10.5px !important;
}

/* MacBook Pro 14 landscape browser view */
@media (min-width:1200px) and (max-height:950px){
  body[data-view="competition"] .dual-chart-title,
  body[data-view="ytd"] .dual-chart-title{
    font-size:17px !important;
    line-height:1.18 !important;
  }

  body[data-view="competition"] .dual-chart-subtitle,
  body[data-view="ytd"] .dual-chart-subtitle{
    font-size:12px !important;
    line-height:1.3 !important;
    min-height:31px !important;
  }

  body[data-view="competition"] .dual-chart-panel,
  body[data-view="ytd"] .dual-chart-panel{
    padding:12px 12px 8px !important;
  }

  body[data-view="competition"] .dual-chart-panel svg,
  body[data-view="ytd"] .dual-chart-panel svg{
    max-height:340px !important;
    min-height:250px !important;
  }

  /* Give the analytical area more visual weight */
  body[data-view="competition"] #chartWrap,
  body[data-view="ytd"] #chartWrap{
    padding:7px 9px 7px !important;
  }

  body[data-view="competition"] .legend,
  body[data-view="ytd"] .legend{
    font-size:11px !important;
    padding:2px 12px 7px !important;
  }

  /* Larger bottom interpretation text across analytical tabs */
  .bottom{
    grid-template-columns:minmax(0,1fr) minmax(0,1fr) !important;
    gap:9px !important;
  }
  .bottom .insight{
    padding:10px 12px !important;
  }
  .bottom .insight h4{
    font-size:10.5px !important;
    margin-bottom:5px !important;
  }
  .bottom .insight p{
    font-size:13px !important;
    line-height:1.34 !important;
  }
  .source{
    font-size:10px !important;
    margin-top:6px !important;
  }

  /* Strategy: larger priority cards and a fuller detail area */
  body[data-view="strategy"] .kpi{
    min-height:74px !important;
  }
  body[data-view="strategy"] .kpi .value{
    font-size:17px !important;
  }
  body[data-view="strategy"] .strategy-detail{
    min-height:300px !important;
    padding:18px 22px !important;
  }
  body[data-view="strategy"] .strategy-field{
    padding:11px 0 !important;
  }
  body[data-view="strategy"] .strategy-field-value{
    font-size:15px !important;
  }
  body[data-view="strategy"] .strategy-field-value.evidence{
    font-size:13.5px !important;
  }
}


/* V36.1: minimal refinements only. Preserve V36 layout and behavior. */

/* Slightly enlarge the two analytical chart areas without changing page structure. */
body[data-view="competition"] .dual-chart-panel svg,
body[data-view="ytd"] .dual-chart-panel svg {
  max-height: 365px !important;
}

/* Use a little more of the available chart card height on single-chart pages. */
body[data-view="markets"] #chartWrap svg,
body[data-view="economy"] #chartWrap svg {
  max-height: 500px !important;
}

/* Slightly larger bottom text for presentation readability. */
.bottom .insight p {
  font-size: 14px !important;
  line-height: 1.38 !important;
}
.bottom .insight h4 {
  font-size: 11.5px !important;
}
.source {
  font-size: 10.8px !important;
}

/* Strategic Recommendations: modestly enlarge content to reduce empty white space. */
body[data-view="strategy"] .strategy-detail {
  min-height: 360px !important;
  padding: 24px 28px !important;
}
body[data-view="strategy"] .strategy-field-value {
  font-size: 16.5px !important;
  line-height: 1.44 !important;
}
body[data-view="strategy"] .strategy-field-value.evidence {
  font-size: 14.5px !important;
  line-height: 1.46 !important;
}

/* Closing: only enlarge the small supporting copy. */
body[data-view="closing"] #chartWrap [style*="font-size:10"],
body[data-view="closing"] #chartWrap [style*="font-size:11"],
body[data-view="closing"] #chartWrap [style*="font-size:12"] {
  font-size: 13.5px !important;
}


/* V36.2: presentation readability. Same V36 structure, larger graphs and text. */
body[data-view="competition"] .dual-chart-panel,
body[data-view="ytd"] .dual-chart-panel{
  padding:12px 12px 8px !important;
}

body[data-view="competition"] .dual-chart-title,
body[data-view="ytd"] .dual-chart-title{
  font-size:17px !important;
  line-height:1.18 !important;
}

body[data-view="competition"] .dual-chart-subtitle,
body[data-view="ytd"] .dual-chart-subtitle{
  font-size:12px !important;
  line-height:1.3 !important;
  min-height:31px !important;
}

body[data-view="competition"] .dual-chart-panel svg,
body[data-view="ytd"] .dual-chart-panel svg{
  max-height:430px !important;
}

.bottom .insight h4{
  font-size:12px !important;
}
.bottom .insight p{
  font-size:14.5px !important;
  line-height:1.38 !important;
}
.source{
  font-size:11px !important;
}
.legend{
  font-size:11.5px !important;
}

/* Use more of the available white space on these two views */
body[data-view="competition"] #chartWrap,
body[data-view="ytd"] #chartWrap{
  padding-top:8px !important;
  padding-bottom:6px !important;
}


/* V36.3 targeted refinements */

/* KPI Overview: chart + evidence snapshot side by side */
body[data-view="performance"] .content{
  display:grid !important;
  grid-template-columns:minmax(0,1fr) 300px !important;
  grid-template-areas:
    "kpis kpis"
    "graph evidence"
    "bottom evidence"
    "source evidence" !important;
  column-gap:10px !important;
  row-gap:8px !important;
  align-items:stretch !important;
}
body[data-view="performance"] #kpis{grid-area:kpis !important}
body[data-view="performance"] .grid{grid-area:graph !important}
body[data-view="performance"] .bottom{grid-area:bottom !important}
body[data-view="performance"] #sourceNote{grid-area:source !important}
body[data-view="performance"] #evidenceCard{
  grid-area:evidence !important;
  margin:0 !important;
  display:block !important;
  padding:14px !important;
  height:100% !important;
}
body[data-view="performance"] #evidenceCard h3{
  font-size:18px !important;
  margin-bottom:12px !important;
}
body[data-view="performance"] .evidence-grid{
  grid-template-columns:1fr !important;
  gap:10px !important;
}
body[data-view="performance"] .evidence-item{
  min-height:118px !important;
  padding:14px !important;
  display:flex !important;
  flex-direction:column !important;
  justify-content:center !important;
}
body[data-view="performance"] .evidence-item .e-title{
  font-size:11px !important;
}
body[data-view="performance"] .evidence-item .e-value{
  font-size:24px !important;
  margin-top:7px !important;
}
body[data-view="performance"] .evidence-item .e-note{
  font-size:12.5px !important;
  line-height:1.35 !important;
}

/* Make KPI overview graph larger and more readable */
body[data-view="performance"] #chartWrap{
  padding:8px 10px 6px !important;
}
body[data-view="performance"] #chartWrap svg{
  min-height:430px !important;
  max-height:520px !important;
}

/* Competitive Analysis: enlarge both graphs further */
body[data-view="competition"] .dual-chart-panel svg{
  min-height:360px !important;
  max-height:470px !important;
}
body[data-view="competition"] .dual-chart-title{
  font-size:18px !important;
}
body[data-view="competition"] .dual-chart-subtitle{
  font-size:12.5px !important;
}

/* Germany Tourism 2026: enlarge both graphs further */
body[data-view="ytd"] .dual-chart-panel svg{
  min-height:360px !important;
  max-height:470px !important;
}
body[data-view="ytd"] .dual-chart-title{
  font-size:18px !important;
}
body[data-view="ytd"] .dual-chart-subtitle{
  font-size:12.5px !important;
}

/* Bottom narrative blocks remain readable */
body[data-view="competition"] .bottom .insight p,
body[data-view="ytd"] .bottom .insight p,
body[data-view="performance"] .bottom .insight p{
  font-size:14.5px !important;
  line-height:1.4 !important;
}

/* Strategic Recommendations: remove Selected Year Context and let remaining content expand */
body[data-view="strategy"] .strategy-detail{
  min-height:380px !important;
  padding:24px 28px !important;
}
body[data-view="strategy"] .strategy-field{
  padding:16px 0 !important;
}
body[data-view="strategy"] .strategy-field-label{
  font-size:12px !important;
}
body[data-view="strategy"] .strategy-field-value{
  font-size:17px !important;
  line-height:1.45 !important;
}
body[data-view="strategy"] .strategy-field-value.evidence{
  font-size:15px !important;
}

</style>
</head>
<body>
<div id="viewportStage"><div class="app" id="app">
  <aside class="sidebar">
    <div class="brand">
      <div class="flag"><i></i><i></i><i></i></div>
      <div><h1>GERMANY</h1><small>TOURISM INTELLIGENCE</small></div>
    </div>
    <div class="analysis-menu">
      <label for="analysisFocus">Analysis Focus</label>
      <select id="analysisFocus" aria-label="Select analysis module">
        <option value="performance">KPI Overview</option>
        <option value="competition">Competitive Analysis</option>
        <option value="markets">Source Market Strategy</option>
        <option value="ytd">Germany Tourism 2026</option>
        <option value="economy">Travel Balance Context</option>
        <option value="strategy">Strategic Recommendations</option>
        <option value="closing">Closing</option>
      </select>
    </div>

    <div class="side-photo" id="sidePhoto">
      <div class="caption"><b id="sidePhotoTitle">Germany</b><span id="sidePhotoCaption">Scenic destinations across Germany</span></div>
    </div>
  </aside>

  <main class="main">
    <section class="hero" id="hero">
      <div class="hero-content">
        <div>
          <div class="eyebrow">Data · Destinations · Opportunities</div>
          <h2 id="heroTitle">Germany Tourism Performance</h2>
          <div class="sub" id="heroSub">Historical performance 2021–2025 with January–June 2026 YTD monitoring</div>
        </div>
        <div id="periodControl" class="period-control">
  <label for="period">Year</label>
  <select class="period" id="period" aria-label="Period">
          <option>2021</option>
          <option>2022</option>
          <option>2023</option>
          <option selected>2024</option>
          <option>2025</option>
          <option>2026 YTD</option>
        </select>
  <span id="yearScopeHint" style="font-size:10px;color:#6d7882;max-width:210px;line-height:1.2"></span>
</div>
      </div>
    </section>

    <div class="content">
      <section class="kpis" id="kpis"></section>

      <section class="grid">
        <article class="card" id="graphCard">
          <div class="card-head">
            <div><h3 id="graphTitle">Germany tourism scale and depth</h3><div class="desc" id="graphDesc">Foreign arrivals, overnight stays and average length of stay</div></div>
          </div>
          <div class="chart-wrap" id="chartWrap"></div>
          <div class="legend" id="legend"></div>
        </article>

        <article class="card photo-card" id="photoCard">
          <div class="slide-counter" id="slideCounter">1 / 9</div>
          <div class="slideshow-controls">
            <button class="slide-btn" id="prevPhoto" type="button" aria-label="Previous German destination">‹</button>
            <button class="slide-btn" id="pausePhoto" type="button" aria-label="Pause slideshow">Ⅱ</button>
            <button class="slide-btn" id="nextPhoto" type="button" aria-label="Next German destination">›</button>
          </div>
          <div class="photo-copy">
            <h3 id="photoTitle">Berlin · Brandenburg Gate</h3>
            <p id="photoText">Germany tourism highlights rotate automatically during the presentation.</p>
          </div>
        </article>
      </section>

      <section class="bottom">
        <article class="card insight" id="insightCard">
          <div class="insight-grid">
            <div><h4>Finding</h4><p id="finding"></p></div>
            <div><h4>Business implication</h4><p id="implication"></p></div>
          </div>
        </article>
        <article class="card insight" id="transitionCard">
          <div><h4>Presentation transition</h4><p id="transition"></p></div>
        </article>
      </section>
      
<section class="evidence-card" id="evidenceCard">
  <h3>Evidence Snapshot</h3>
  <div class="evidence-grid" id="evidenceGrid"></div>
</section>

<div class="source" id="sourceNote"></div>
    </div>
  </main>
</div></div>

<div class="fit-toolbar" id="fitToolbar" aria-label="Dashboard size controls">
  <button type="button" id="zoomOut" title="Make dashboard smaller">−</button>
  <button type="button" id="fitScreen" title="Fit dashboard to screen">Fit</button>
  <span class="fit-label" id="fitLabel">100%</span>
  <button type="button" id="zoomIn" title="Make dashboard larger">+</button>
</div>
<div class="tooltip" id="tooltip"></div>

<script>
const IMG_BERLIN="https://images.unsplash.com/photo-1570862687812-8b841fad0733?auto=format&fit=crop&fm=jpg&q=88&w=2200";
const IMG_CASTLE="https://images.unsplash.com/photo-1780474980674-b03c882a8d34?auto=format&fit=crop&fm=jpg&q=88&w=2200";
const tourismPhotos=[
 {name:'Neuschwanstein Castle · Bavaria',caption:'Fairytale castle and Alpine scenery in Bavaria.',url:'https://images.unsplash.com/photo-1780474980674-b03c882a8d34?auto=format&fit=crop&fm=jpg&q=88&w=2200'},
 {name:'Brandenburg Gate · Berlin',caption:"Berlin's landmark monument and historic urban tourism icon.",url:'https://images.unsplash.com/photo-1570862687812-8b841fad0733?auto=format&fit=crop&fm=jpg&q=88&w=2200'},
 {name:'Heidelberg Castle · Baden-Württemberg',caption:'Romantic castle landscape overlooking historic Heidelberg.',url:'https://images.unsplash.com/photo-1744049891187-99ce6f6bca57?auto=format&fit=crop&fm=jpg&q=88&w=2200'},
 {name:'Königssee · Bavaria',caption:'A dramatic Alpine lake surrounded by mountain scenery.',url:'https://images.unsplash.com/photo-1732205065140-e6dfecc87eb8?auto=format&fit=crop&fm=jpg&q=88&w=2200'}
];
let photoIndex=0;
let photoTimer=null;
let slideshowPaused=false;

function showTourismPhoto(index,manual=false){
 photoIndex=(index+tourismPhotos.length)%tourismPhotos.length;
 const p=tourismPhotos[photoIndex];
 photoCard.style.opacity=".72";
 hero.style.opacity=".88";
 setTimeout(()=>{
   photoCard.style.backgroundImage=`url("${p.url}")`;
   hero.style.backgroundImage=`url("${p.url}")`;
   sidePhoto.style.backgroundImage=`url("${p.url}")`;
   document.getElementById("photoTitle").textContent=p.name;
   document.getElementById("photoText").textContent=p.caption;
   document.getElementById("slideCounter").textContent=`${photoIndex+1} / ${tourismPhotos.length}`;
   const spt=document.getElementById("sidePhotoTitle");
   const spc=document.getElementById("sidePhotoCaption");
   if(spt) spt.textContent=p.name;
   if(spc) spc.textContent=p.caption;
   photoCard.style.opacity="1";
   hero.style.opacity="1";
 },140);
 if(manual && !slideshowPaused) restartPhotoTimer();
}

function restartPhotoTimer(){
 clearInterval(photoTimer);
 photoTimer=setInterval(()=>showTourismPhoto(photoIndex+1),5000);
}

function togglePhotoSlideshow(){
 slideshowPaused=!slideshowPaused;
 const btn=document.getElementById("pausePhoto");
 btn.textContent=slideshowPaused?"▶":"Ⅱ";
 btn.setAttribute("aria-label",slideshowPaused?"Resume slideshow":"Pause slideshow");
 if(slideshowPaused) clearInterval(photoTimer);
 else restartPhotoTimer();
}


const annual=[
 {year:2021,arrivals:11.66,nights:30.73,los:2.64},
 {year:2022,arrivals:28.38,nights:67.62,los:2.38},
 {year:2023,arrivals:34.71,nights:80.38,los:2.32},
 {year:2024,arrivals:37.42,nights:84.79,los:2.27},
 {year:2025,arrivals:37.13,nights:83.08,los:2.24}
];

const views={
closing:{
 title:"Closing",
 sub:"From Tourism Scale to Greater Tourism Depth",
 photo:IMG_CASTLE,
 photoTitle:"Germany · The Travel Destination",
 photoText:"Germany already has strong tourism scale. The strategic opportunity is to convert that scale into longer stays and greater tourism value.",
 kpis:[],
 graphTitle:"",
 graphDesc:"",
 chart:"closing",
 finding:"Germany does not primarily have a tourism scale problem. It has an opportunity to convert its already strong visitor volume into greater tourism depth.",
 implication:"Thank you.",
 transition:"For questions and queries, please scan the QR code.",
 source:"Germany tourism competitiveness analysis · Jan Noel L. Vero"
},
performance:{
 title:"Germany Tourism Performance",
 sub:"Historical performance 2021–2025 with January–June 2026 YTD monitoring",
 photo:IMG_BERLIN, photoTitle:"Berlin · Brandenburg Gate", photoText:"Germany enters the analysis as one of the EU's largest international tourism destinations.",
 kpis:[
  ["2024 Foreign Arrivals","37.42 M","4th in EU27"],
  ["2024 Foreign Nights","84.79 M","7th in EU27"],
  ["2024 Avg. Stay","2.27 nights","21st in EU27"],
  ["H1 2026 Arrivals","+2.04%","vs. H1 2025"]
 ],
 graphTitle:"Recovery in scale, pressure on stay duration", graphDesc:"Observed annual foreign tourism performance",
 chart:"annual",
 finding:"Germany recovered strongly in visitor scale, but average length of stay fell from 2.64 nights in 2021 to 2.24 nights in 2025.",
 implication:"Growth should not be judged through arrivals alone. Germany needs to convert a strong visitor base into more overnight-stay depth.",
 transition:"Germany is large in scale. The next question is whether that scale translates into a strong competitive position.",
 source:"Verified project output · Eurostat accommodation statistics · 2026 YTD = January–June."
},
competition:{
 title:"Germany’s EU Tourism Competitiveness: Strong Scale, Limited Stay Duration",
 sub:"",
 photo:IMG_CASTLE, photoTitle:"Germany · Competitive Position", photoText:"Germany is a high-volume EU destination, but its average stay remains comparatively short.",
 kpis:[
  ["EU Arrivals Rank","#4","2024 foreign arrivals"],
  ["EU Nights Rank","#7","2024 foreign overnight stays"],
  ["Average Stay","2.27","nights"],
  ["EU LOS Rank","#21","among EU27"]
 ],
 graphTitle:"Competitive Position and Tourism-Depth Opportunity",
 graphDesc:"2024 EU27 benchmark and illustrative stay-duration counterfactuals",
 chart:"competition",
 finding:"Germany combines strong international tourism scale with below-average stay duration. It ranks #4 in foreign arrivals but only #21 in average length of stay among EU27 countries.",
 implication:"The central competitiveness opportunity is not simply attracting more visitors. Extending existing trips could materially increase overnight volume. Matching the EU27 median LOS would imply about 8.39 million additional nights at 2024 arrival volume.",
 transition:"The next question is which source markets offer the strongest opportunities for growth, retention and stay extension.",
 source:"Eurostat 2024 accommodation statistics. Counterfactuals hold Germany's 37.42M foreign arrivals constant and apply alternative LOS benchmarks; they are illustrative, not forecasts."
},
markets:{
 title:"Growth and Strategy",
 sub:"Source Market Strategy",
 photo:IMG_BERLIN, photoTitle:"Germany's International Gateways", photoText:"Source-market strategy should reflect scale, growth and stay behavior rather than one uniform acquisition approach.",
 kpis:[
  ["Core Growth","US + UK","Established scale + growth"],
  ["Core Retention","NL + Poland","Protect existing scale"],
  ["China Growth","+40.55%","2023 → 2024 nights"],
  ["Türkiye Growth","+17.95%","2023 → 2024 nights"]
 ],
 graphTitle:"Source-Market Growth and Stay Strategy", graphDesc:"Growth, stay duration, overnight scale and strategic role",
 chart:"markets",
 finding:"The United States and United Kingdom are core growth markets. Netherlands and Poland are core retention markets. China and Türkiye are emerging growth opportunities.",
 implication:"Germany should use differentiated acquisition, retention and stay-extension strategies rather than one international marketing approach.",
 transition:"Market growth matters, but tourism competitiveness also has an economic dimension.",
 source:"Source-market analysis based on 2024 overnight scale, 2023–2024 growth, average stay and strategic role."
},
ytd:{
 title:"Germany Tourism 2026: Positive Arrival Momentum, but Stay Duration Remains a Constraint",
 sub:"",
 photo:IMG_BERLIN, photoTitle:"Germany Tourism 2026", photoText:"Current momentum and conversion of arrivals into overnight stays.",
 kpis:[
  ["H1 Arrivals Growth","+2.04%","vs H1 2025"],
  ["H1 Nights Growth","+0.74%","vs H1 2025"],
  ["Avg. Stay Change","−1.27%","vs H1 2025"],
  ["2026 Baseline","≈83.34 M nights","Seasonal-naive scenario"]
 ],
 graphTitle:"2026 Growth in Arrivals Has Not Fully Translated into Longer Stays",
 graphDesc:"January–June 2026 year-on-year change in foreign tourism indicators",
 chart:"ytd",
 finding:"International arrivals are growing faster than overnight stays, while average stay has declined.",
 implication:"Germany is generating visitor momentum, but the current challenge is converting that growth into tourism depth.",
 transition:"This leads to the question of how Germany should manage growth and source markets.",
 source:"2026 YTD = January–June observed data. The full-year value is a seasonal-naive baseline scenario, not a definitive forecast."
},
economy:{
 title:"Travel Economic Context",
 sub:"Inbound receipts are growing, while Germany remains a major outbound travel economy",
 photo:IMG_CASTLE, photoTitle:"Tourism Value", photoText:"Economic context complements accommodation statistics but represents a different statistical system.",
 kpis:[
  ["2021 Receipts","€18.8 B","International travel"],
  ["2025 Receipts","€37.8 B","Provisional"],
  ["2025 Expenditure","€114.6 B","German residents abroad"],
  ["2025 Travel Balance","−€76.8 B","Broader travel flows"]
 ],
 graphTitle:"Travel receipts versus expenditure", graphDesc:"Balance-of-payments travel context · 2021 and 2025",
 chart:"economy",
 finding:"Inbound travel receipts roughly doubled from €18.8B in 2021 to €37.8B in 2025, while outbound travel expenditure rose to €114.6B.",
 implication:"The negative travel balance is economic context, not evidence that Germany's tourism industry is unprofitable. Accommodation and balance-of-payments statistics measure different systems.",
 transition:"The descriptive evidence shows a declining stay-duration pattern. The next question is whether that decline is statistically supported.",
 source:"Eurostat Balance of Payments travel data. 2025 values are provisional."
},
statistics:{
 title:"Statistical Evidence",
 sub:"Seasonality-adjusted trend in Germany's international average length of stay",
 photo:IMG_BERLIN, photoTitle:"Evidence Before Recommendation", photoText:"The statistical result supports a declining stay-duration trend, but it does not establish causality.",
 kpis:[
  ["Annualized Trend","−0.0602 nights","per year"],
  ["Robust Test Stat","−5.14","HAC(3)"],
  ["One-sided p","0.000005","Reject H₀"],
  ["95% CI","−0.0840 to −0.0365","annualized"]
 ],
 graphTitle:"Statistically supported decline in average stay", graphDesc:"Seasonality-adjusted regression with HAC standard errors",
 chart:"statistics",
 finding:"The null hypothesis was rejected. There is statistically significant evidence that international average length of stay declined over the tested period.",
 implication:"The declining stay-duration pattern is strong enough to support management attention, while remaining an association over time rather than proof of a causal mechanism.",
 transition:"With the competitive, market and statistical evidence aligned, the analysis can now move to action.",
 source:"Primary HAC(3) regression result. Statistical association, not causality."
},
strategy:{
 title:"Strategic Priorities",
 sub:"Convert Germany's tourism scale into greater stay depth and value",
 photo:IMG_CASTLE, photoTitle:"Germany · From Scale to Depth", photoText:"The final recommendation is to protect visitor scale while increasing tourism depth.",
 kpis:[
  ["Priority 1","Increase stay duration","Primary KPI · LOS"],
  ["Priority 2","Differentiate markets","Primary KPI · source nights"],
  ["Priority 3","Convert growth to depth","Nights vs arrivals growth"],
  ["Priority 4","Monitor leading indicators","YTD + seasonal baseline"]
 ],
 graphTitle:"Evidence → management action", graphDesc:"Four strategic priorities from the completed project",
 chart:"strategy",
 finding:"Germany's competitive challenge is not primarily visibility or visitor scale. It is converting an already strong international visitor position into greater overnight-stay depth and tourism value.",
 implication:"Protect established demand, develop suitable stay-extension opportunities, differentiate source-market strategy and monitor whether arrival growth converts into overnight growth.",
 transition:"This returns the presentation to the central conclusion: Germany is strong in scale, but its largest opportunity is tourism depth.",
 source:"Integrated strategic diagnosis from the completed Germany Tourism Competitiveness project."
}
};

const app=document.getElementById("app"), kpis=document.getElementById("kpis"), chartWrap=document.getElementById("chartWrap");
const tooltip=document.getElementById("tooltip");
const hero=document.getElementById("hero"), photoCard=document.getElementById("photoCard"), sidePhoto=document.getElementById("sidePhoto");

function flash(el){
 document.querySelectorAll(".card,.kpi").forEach(x=>x.classList.add("dim"));
 el.classList.remove("dim"); el.classList.remove("flash"); void el.offsetWidth; el.classList.add("flash");
 setTimeout(()=>document.querySelectorAll(".card,.kpi").forEach(x=>x.classList.remove("dim")),650);
}

function barChart(items,max,labelSuffix=""){
 const W=760,H=310,L=180,R=45,T=24,B=35,plot=W-L-R;
 let out=`<svg viewBox="0 0 ${W} ${H}" aria-label="Bar chart">`;
 items.forEach((d,i)=>{
  let y=T+i*52, w=(d.value/max)*plot;
  out+=`<text x="${L-12}" y="${y+22}" text-anchor="end" font-size="14" fill="#33414d">${d.name}</text>
  <rect x="${L}" y="${y}" width="${plot}" height="28" rx="5" fill="#edf1f4"></rect>
  <rect class="mark" data-tip="${d.tip||d.name+': '+d.value+labelSuffix}" x="${L}" y="${y}" width="${w}" height="28" rx="5" fill="${d.color||'#d71920'}"></rect>
  <text x="${Math.min(L+w+9,W-35)}" y="${y+20}" font-size="13" font-weight="700" fill="#17222c">${d.display||d.value+labelSuffix}</text>`;
 });
 out+=`</svg>`; return out;
}
function annualChart(){
 const W=760,H=340,L=55,R=40,T=25,B=45,pw=W-L-R,ph=H-T-B;
 const xs=i=>L+i*(pw/(annual.length-1));
 const yN=v=>T+ph-(v/90)*ph;
 const yL=v=>T+ph-((v-2.0)/0.8)*ph;
 let pArr=annual.map((d,i)=>`${xs(i)},${yN(d.arrivals)}`).join(" ");
 let pNight=annual.map((d,i)=>`${xs(i)},${yN(d.nights)}`).join(" ");
 let pLos=annual.map((d,i)=>`${xs(i)},${yL(d.los)}`).join(" ");
 let s=`<svg viewBox="0 0 ${W} ${H}" aria-label="Germany tourism annual trend">
 <line x1="${L}" y1="${T+ph}" x2="${W-R}" y2="${T+ph}" stroke="#cad4dc"/><line x1="${L}" y1="${T}" x2="${L}" y2="${T+ph}" stroke="#cad4dc"/>
 <polyline points="${pNight}" fill="none" stroke="#1565c0" stroke-width="4"/><polyline points="${pArr}" fill="none" stroke="#d71920" stroke-width="4"/><polyline points="${pLos}" fill="none" stroke="#f1b514" stroke-width="4"/>`;
 annual.forEach((d,i)=>{
  s+=`<text x="${xs(i)}" y="${H-17}" text-anchor="middle" font-size="12" fill="#596a78">${d.year}</text>
  <circle class="mark" data-tip="${d.year} foreign arrivals: ${d.arrivals}M" cx="${xs(i)}" cy="${yN(d.arrivals)}" r="5" fill="#d71920"/>
  <circle class="mark" data-tip="${d.year} foreign overnight stays: ${d.nights}M" cx="${xs(i)}" cy="${yN(d.nights)}" r="5" fill="#1565c0"/>
  <circle class="mark" data-tip="${d.year} average length of stay: ${d.los} nights" cx="${xs(i)}" cy="${yL(d.los)}" r="5" fill="#f1b514"/>`;
 });
 s+=`</svg>`; return s;
}
function marketChart(){
 const markets=[
  {name:"United States",growth:8.6,los:2.55,nights:11.87,role:"Core Growth Market",label:"United States"},
  {name:"United Kingdom",growth:7.2,los:2.48,nights:6.7,role:"Core Growth Market",label:"United Kingdom"},
  {name:"Netherlands",growth:-1.2,los:2.28,nights:10.4,role:"Core Retention Market",label:"Netherlands"},
  {name:"Poland",growth:0.16,los:2.832,nights:4.167089,role:"Core Retention Market",label:"Poland"},
  {name:"China",growth:40.55,los:2.17,nights:2.5,role:"Emerging Growth Market",label:"China"},
  {name:"Türkiye",growth:17.95,los:2.35,nights:1.8,role:"Emerging Growth Market",label:"Türkiye"},
  {name:"Romania",growth:2.0,los:3.46,nights:1.6,role:"Long-Stay Niche",label:"Romania"},
  {name:"Switzerland and Liechtenstein",growth:3.1,los:1.80,nights:5.0,role:"Stay Extension Opportunity",label:""},
  {name:"Austria",growth:2.4,los:1.82,nights:4.2,role:"Stay Extension Opportunity",label:""},
  {name:"France",growth:4.1,los:2.15,nights:4.8,role:"Stay Extension Opportunity",label:""}
 ];
 const roleColors={
  "Core Growth Market":"#337caf",
  "Core Retention Market":"#f28e00",
  "Emerging Growth Market":"#ef5350",
  "Long-Stay Niche":"#43a047",
  "Stay Extension Opportunity":"#a66aa5"
 };
 const W=820,H=430,L=62,R=30,T=24,B=48;
 const x=v=>L+((v+2.2)/(44))*(W-L-R);
 const y=v=>T+((3.65-v)/(3.65-1.45))*(H-T-B);
 const r=n=>5+Math.sqrt(n/11.87)*10;
 const refLos=2.27;

 const marks=markets.map(m=>{
  const tip=`Strategic Market Label: ${m.label||""}\nSource Market Name: ${m.name}\nStrategic Role: ${m.role}\nAvg. Avg Length Of Stay: ${m.los.toFixed(3)}\nAvg. Growth 2023 2024 Pct: ${m.growth.toFixed(2)}\nForeign Nights: ${Math.round(m.nights*1000000).toLocaleString("en-US")}\nParameter 1: 0`;
  return `<circle class="mark" cx="${x(m.growth)}" cy="${y(m.los)}" r="${r(m.nights)}"
    fill="${roleColors[m.role]||'#777'}" fill-opacity=".82" stroke="#fff" stroke-width="1.5"
    data-tip="${tip.replace(/\n/g,'&#10;')}"></circle>
   ${m.label?`<text x="${x(m.growth)}" y="${y(m.los)-r(m.nights)-5}" text-anchor="middle" font-size="10" fill="#737d85">${m.label}</text>`:""}`;
 }).join("");

 const legend=Object.entries(roleColors).map(([role,c],i)=>
  `<rect x="665" y="${50+i*24}" width="12" height="12" fill="${c}"/><text x="683" y="${60+i*24}" font-size="10">${role}</text>`
 ).join("");

 return `<div>
  <div style="font-size:18px;font-weight:800;margin:0 0 2px">Germany's Source Markets Require Different Growth and Stay Strategies</div>
  <div style="font-size:12px;color:#687580;margin-bottom:4px">2024 stay duration vs. 2023–2024 growth; bubble size represents foreign overnight stays</div>
  <svg viewBox="0 0 ${W} ${H}" style="width:100%;height:auto">
   ${[-0,10,20,30,40].map(v=>`<line x1="${x(v)}" y1="${T}" x2="${x(v)}" y2="${H-B}" stroke="#edf0f2"/><text x="${x(v)}" y="${H-25}" text-anchor="middle" font-size="10">${v}</text>`).join("")}
   ${[1.5,2.0,2.5,3.0,3.5].map(v=>`<line x1="${L}" y1="${y(v)}" x2="${W-R}" y2="${y(v)}" stroke="#edf0f2"/><text x="${L-8}" y="${y(v)+4}" text-anchor="end" font-size="10">${v.toFixed(1)}</text>`).join("")}
   <line x1="${x(0)}" y1="${T}" x2="${x(0)}" y2="${H-B}" stroke="#8d969e" stroke-width="1.4"/>
   <line x1="${L}" y1="${y(refLos)}" x2="${W-R}" y2="${y(refLos)}" stroke="#8d969e" stroke-width="1.4"/>
   <text x="${x(0)+4}" y="${H-B-5}" font-size="10" fill="#727b83">No growth</text>
   <text x="${L+4}" y="${y(refLos)-6}" font-size="10" fill="#727b83">Germany LOS benchmark</text>
   ${marks}
   ${legend}
   <text x="${(L+W-R)/2}" y="${H-6}" text-anchor="middle" font-size="11">Growth 2023–2024 (%)</text>
   <text transform="translate(14 ${(T+H-B)/2}) rotate(-90)" text-anchor="middle" font-size="11">Avg. Length of Stay</text>
  </svg>
 </div>`;
}

function tourism2026Chart(){
 const momentum=[
  ["January",3.739578323018885,0.8246865201175423,-2.809816513640615],
  ["February",4.16586804546134,2.595516389029467,-1.5075491481974987],
  ["March",2.0421742290484,-0.4459273570753878,-2.4383071067643822],
  ["April",1.2512445173963351,0.6098260359202818,-0.6334919482059646],
  ["May",0.7330394326731983,-0.5089820039269212,-1.2329831836656149],
  ["June",1.67640579155227,1.6303623281162944,-0.0452843145639557]
 ];
 const outlook=[
  ["January",4277203,"Observed"],["February",5078278,"Observed"],["March",5197078,"Observed"],
  ["April",6596278,"Observed"],["May",7446648,"Observed"],["June",7653746,"Observed"],
  ["July",10358796,"Projected"],["August",9689151,"Projected"],["September",7674317,"Projected"],
  ["October",7195093,"Projected"],["November",5590732,"Projected"],["December",6582972,"Projected"]
 ];
 const W=820,H1=360,L=62,R=28,T=28,B=48;
 const x=i=>L+i*((W-L-R)/(momentum.length-1));
 const y=v=>T+((4.6-v)/(4.6+3.2))*(H1-T-B);
 const series=[
  {idx:1,name:"Foreign Arrivals",c:"#f57c00"},
  {idx:2,name:"Foreign Nights",c:"#f1b514"},
  {idx:3,name:"Average Length of Stay",c:"#a66f52"}
 ];
 const paths=series.map(s=>`<path d="${momentum.map((d,i)=>`${i?'L':'M'} ${x(i)} ${y(d[s.idx])}`).join(' ')}" fill="none" stroke="${s.c}" stroke-width="3"/>`).join("");
 const pts=series.map(s=>momentum.map((d,i)=>{
   const tip=`${d[0]} 2026\n\nIndicator: ${s.name}\nYear-on-year change: ${d[s.idx].toFixed(2)}%\n\nCompared with the same month in 2025`;
   return `<circle class="mark" cx="${x(i)}" cy="${y(d[s.idx])}" r="5" fill="${s.c}" data-tip="${tip.replace(/\n/g,'&#10;')}"></circle>`;
 }).join("")).join("");

 const H2=350,L2=62,R2=28,T2=28,B2=48;
 const xx=i=>L2+i*((W-L2-R2)/(outlook.length-1));
 const yy=v=>T2+((11-v/1e6)/11)*(H2-T2-B2);
 const obs=outlook.slice(0,6),proj=outlook.slice(5);
 const path=(arr,offset,c)=>`<path d="${arr.map((d,j)=>`${j?'L':'M'} ${xx(j+offset)} ${yy(d[1])}`).join(' ')}" fill="none" stroke="${c}" stroke-width="3"/>`;
 const outlookPts=outlook.map((d,i)=>{
   const tip=`${d[0]} 2026\n\nForeign overnight stays: ${Math.round(d[1]).toLocaleString("en-US")}\nStatus: ${d[2]}\n\n${d[2]==="Projected"?"Projected values use the seasonal-naive scenario based on the corresponding 2025 month.":"Observed value."}`;
   return `<circle class="mark" cx="${xx(i)}" cy="${yy(d[1])}" r="5" fill="${d[2]==="Observed"?"#f1b514":"#f57c00"}" data-tip="${tip.replace(/\n/g,'&#10;')}"></circle>`;
 }).join("");

 return `<div class="dual-chart-grid">
  <section class="dual-chart-panel">
    <div class="dual-chart-title">2026 Growth in Arrivals Has Not Fully Translated into Longer Stays</div>
    <div class="dual-chart-subtitle">Monthly year-on-year change in Germany's foreign tourism indicators, January–June 2026</div>
    <svg viewBox="0 0 ${W} ${H1}" style="width:100%;height:auto">
     ${[-2,0,2,4].map(v=>`<line x1="${L}" y1="${y(v)}" x2="${W-R}" y2="${y(v)}" stroke="${v===0?'#8c949b':'#edf0f2'}"/><text x="${L-8}" y="${y(v)+4}" text-anchor="end" font-size="11">${v}</text>`).join("")}
     ${momentum.map((d,i)=>`<text x="${x(i)}" y="${H1-12}" text-anchor="middle" font-size="11">${d[0]}</text>`).join("")}
     ${paths}${pts}
     <text transform="translate(14 ${(T+H1-B)/2}) rotate(-90)" text-anchor="middle" font-size="10">Year-on-Year Change (%)</text>
    </svg>
  </section>

  <section class="dual-chart-panel">
    <div class="dual-chart-title">2026 Seasonal Baseline Suggests Broadly Stable Full-Year Foreign Nights</div>
    <div class="dual-chart-subtitle">January–June observed; July–December seasonal-naive scenario based on corresponding 2025 monthly levels</div>
    <svg viewBox="0 0 ${W} ${H2}" style="width:100%;height:auto">
     ${[0,2,4,6,8,10].map(v=>`<line x1="${L2}" y1="${yy(v*1e6)}" x2="${W-R2}" y2="${yy(v*1e6)}" stroke="#edf0f2"/><text x="${L2-8}" y="${yy(v*1e6)+4}" text-anchor="end" font-size="10">${v}M</text>`).join("")}
     ${path(obs,0,"#f1b514")}
     ${path(proj,5,"#f57c00")}
     ${outlookPts}
     ${outlook.map((d,i)=>`<text x="${xx(i)}" y="${H2-12}" text-anchor="middle" font-size="8">${d[0].slice(0,3)}</text>`).join("")}
     <text transform="translate(14 ${(T2+H2-B2)/2}) rotate(-90)" text-anchor="middle" font-size="10">Foreign Nights Value</text>
    </svg>
  </section>
 </div>`;
}
function economyChart(){
 const selectedYear=Number(document.getElementById("period")?.value)||2024;
 const rows=[
  {year:2021,receipts:18827,expenditure:43126,balance:-24300,status:"Final"},
  {year:2022,receipts:30257,expenditure:85019,balance:-54762,status:"Final"},
  {year:2023,receipts:34992,expenditure:106642,balance:-71650,status:"Final"},
  {year:2024,receipts:37055,expenditure:106822,balance:-69767,status:"Final"},
  {year:2025,receipts:37772,expenditure:114594,balance:-76822,status:"Provisional"}
 ];
 const W=820,H=410,L=70,R=32,T=25,B=48,max=120000;
 const x=i=>L+i*((W-L-R)/(rows.length-1));
 const y=v=>T+(max-v)/max*(H-T-B);
 const line=(key,c)=>`<path d="${rows.map((d,i)=>`${i?'L':'M'} ${x(i)} ${y(d[key])}`).join(' ')}" fill="none" stroke="${c}" stroke-width="3"/>`;
 const points=(key,c,label)=>rows.map((d,i)=>{
   const tip=`Year: ${d.year}\n\nTravel Expenditure Mio Eur: ${d.expenditure.toLocaleString("en-US")}\nTravel Receipts Mio Eur: ${d.receipts.toLocaleString("en-US")}\n\nData Status: ${d.status}`;
   return `<circle class="mark" cx="${x(i)}" cy="${y(d[key])}" r="${d.year===selectedYear?8:4.5}" fill="${c}" stroke="${d.year===selectedYear?'#0c2d4c':'none'}" stroke-width="${d.year===selectedYear?2.5:0}" data-tip="${tip.replace(/\n/g,'&#10;')}"></circle>`;
 }).join("");
 return `<div>
  <div style="font-size:18px;font-weight:800;margin:0 0 2px">Germany's Travel Expenditure Has Grown Much Faster Than Travel Receipts</div>
  <div style="font-size:12px;color:#687580;margin-bottom:5px">Travel balance-of-payments flows, 2021–2025; 2025 data provisional</div>
  <svg viewBox="0 0 ${W} ${H}" style="width:100%;height:auto">
   ${[0,20000,40000,60000,80000,100000,120000].map(v=>`<line x1="${L}" y1="${y(v)}" x2="${W-R}" y2="${y(v)}" stroke="#edf0f2"/><text x="${L-9}" y="${y(v)+4}" text-anchor="end" font-size="10">${v/1000}K</text>`).join("")}
   ${rows.map((d,i)=>`<line x1="${x(i)}" y1="${T}" x2="${x(i)}" y2="${H-B}" stroke="#f1f3f5"/><text x="${x(i)}" y="${H-17}" text-anchor="middle" font-size="10">${d.year}</text>`).join("")}
   ${line("expenditure","#f1b514")}
   ${line("receipts","#f57c00")}
   ${points("expenditure","#f1b514","Travel Expenditure")}
   ${points("receipts","#f57c00","Travel Receipts")}
   <text transform="translate(17 ${(T+H-B)/2}) rotate(-90)" text-anchor="middle" font-size="11">Million EUR</text>
  </svg>
 </div>`;
}

function closingChart(){
 return `<div class="closing-photo-stage" style="position:relative;min-height:570px;border-radius:14px;overflow:hidden;background:#0c263b">
   <div id="closingPhoto" style="position:absolute;inset:0;background-image:url('${tourismPhotos[photoIndex].url}');background-size:cover;background-position:center;transition:opacity .35s ease"></div>
   <div style="position:absolute;inset:0;background:linear-gradient(90deg,rgba(5,22,38,.08) 45%,rgba(255,255,255,.92) 70%,rgba(255,255,255,.98) 100%)"></div>

   <div style="position:absolute;left:26px;top:24px;color:white;text-shadow:0 1px 4px rgba(0,0,0,.55)">
     <div style="font-size:18px;font-weight:800" id="closingPhotoTitle">${tourismPhotos[photoIndex].name}</div>
     <div style="font-size:12px;margin-top:4px" id="closingPhotoCaption">${tourismPhotos[photoIndex].caption}</div>
   </div>

   <button type="button" id="closingPrev" aria-label="Previous destination"
     style="position:absolute;left:22px;top:48%;width:46px;height:46px;border-radius:50%;border:0;background:rgba(5,20,34,.66);color:white;font-size:30px;cursor:pointer">‹</button>
   <button type="button" id="closingNext" aria-label="Next destination"
     style="position:absolute;left:61%;top:48%;width:46px;height:46px;border-radius:50%;border:0;background:rgba(5,20,34,.66);color:white;font-size:30px;cursor:pointer">›</button>

   <div style="position:absolute;left:26px;bottom:28px;color:white;background:rgba(5,20,34,.58);padding:14px 17px;border-radius:9px">
     <div style="font-family:Georgia,serif;font-style:italic;font-size:24px;line-height:1.05">Timeless places.<br>New perspectives.</div>
     <div id="closingDots" style="margin-top:12px"></div>
   </div>

   <div style="position:absolute;right:2.5%;top:9%;width:35%;text-align:center;color:#0c2d4c">
     <div style="font-size:25px;font-weight:800">Germany</div>
     <div style="font-size:16px;font-style:italic;margin-bottom:54px">The travel destination</div>
     <div style="font-family:Georgia,serif;font-style:italic;font-size:48px;line-height:1">Vielen Dank!</div>
     <div style="width:52px;height:3px;background:#f2b705;margin:22px auto"></div>
     <div style="font-family:Georgia,serif;font-style:italic;font-size:21px;line-height:1.22">
       Deutschland.<br>Mehr als ein Reiseziel.<br>Ein bleibender Eindruck.
     </div>
     <div style="margin:18px auto 0;display:flex;align-items:center;justify-content:center;gap:16px;text-align:left">
       <img src="..." alt="Jan Noel Vero contact QR code" style="width:230px;height:auto;border-radius:10px;background:white;padding:8px;box-shadow:0 3px 14px rgba(0,0,0,.14)">
       <div style="font-size:12px;line-height:1.4;color:#173a5e">
         <b style="font-size:13px">Bleiben wir in Kontakt.</b><br>
         Scannen Sie den QR-Code,<br>um mein Portfolio zu sehen.
       </div>
     </div>
     <div style="margin-top:42px;font-size:10px;letter-spacing:4px;color:#607287">CULTURE&nbsp;&nbsp; NATURE&nbsp;&nbsp; PEOPLE&nbsp;&nbsp; OPPORTUNITIES</div>
     <div style="display:flex;width:150px;height:4px;margin:14px auto 0">
       <span style="flex:1;background:#111"></span><span style="flex:1;background:#e21b23"></span><span style="flex:1;background:#f2b705"></span>
     </div>
   </div>
 </div>`;
}

function bindClosingSlideshow(){
 const stage=document.getElementById("closingPhoto");
 if(!stage) return;
 const render=()=>{
   const p=tourismPhotos[photoIndex];
   stage.style.opacity=".45";
   setTimeout(()=>{
     stage.style.backgroundImage=`url("${p.url}")`;
     const t=document.getElementById("closingPhotoTitle");
     const c=document.getElementById("closingPhotoCaption");
     if(t)t.textContent=p.name;
     if(c)c.textContent=p.caption;
     stage.style.opacity="1";
     const dots=document.getElementById("closingDots");
     if(dots)dots.innerHTML=tourismPhotos.map((_,i)=>`<span data-ci="${i}" style="display:inline-block;width:9px;height:9px;border-radius:50%;margin-right:7px;background:${i===photoIndex?'#fff':'rgba(255,255,255,.45)'};cursor:pointer"></span>`).join("");
     document.querySelectorAll("#closingDots [data-ci]").forEach(d=>d.onclick=()=>{photoIndex=Number(d.dataset.ci);render();});
   },120);
 };
 const prev=document.getElementById("closingPrev");
 const next=document.getElementById("closingNext");
 if(prev)prev.onclick=()=>{photoIndex=(photoIndex-1+tourismPhotos.length)%tourismPhotos.length;render();};
 if(next)next.onclick=()=>{photoIndex=(photoIndex+1)%tourismPhotos.length;render();};
 render();
}
function statsChart(){
 const min=-0.11,max=0.03,W=760,H=260,L=55,R=40,y=125,sc=v=>L+(v-min)/(max-min)*(W-L-R);
 return `<svg viewBox="0 0 ${W} ${H}" aria-label="Confidence interval and trend estimate">
 <line x1="${L}" y1="${y}" x2="${W-R}" y2="${y}" stroke="#9aa8b5" stroke-width="2"/>
 <line x1="${sc(0)}" y1="55" x2="${sc(0)}" y2="205" stroke="#17222c" stroke-dasharray="5 5"/>
 <line x1="${sc(-0.084)}" y1="${y}" x2="${sc(-0.0365)}" y2="${y}" stroke="#1565c0" stroke-width="14" stroke-linecap="round"/>
 <circle class="mark" data-tip="Annualized trend estimate: −0.0602 nights per year" cx="${sc(-0.0602)}" cy="${y}" r="10" fill="#d71920"/>
 <text x="${sc(-0.084)}" y="95" text-anchor="middle" font-size="12">−0.0840</text>
 <text x="${sc(-0.0365)}" y="95" text-anchor="middle" font-size="12">−0.0365</text>
 <text x="${sc(0)+8}" y="70" font-size="12">No trend</text>
 <text x="${W/2}" y="225" text-anchor="middle" font-size="13" fill="#667484">95% annualized confidence interval · entirely below zero</text>
 </svg>`;
}
function strategyChart(){
 window.strategyPriorities=[
  {
   strategic:"Increase Stay Duration",
   direction:"Develop trip-extension, multi-city and regional itinerary propositions that encourage visitors to add nights to their Germany trip.",
   evidence:"Germany ranks 21st in EU27 for average length of stay despite ranking 4th in foreign arrivals. In 2024, Germany's LOS was 2.27 nights versus the EU median of 2.49 nights."
  },
  {
   strategic:"Differentiate Source-Market Strategy",
   direction:"Use separate growth, retention, stay-extension and niche-market strategies instead of applying one uniform approach across all international markets.",
   evidence:"Source markets show materially different patterns in scale, growth and stay behavior. The US and UK are Core Growth markets, the Netherlands and Poland are Core Retention markets, China and Türkiye are Emerging Growth markets, and Romania and Czech Republic are Long-Stay Niche markets."
  },
  {
   strategic:"Convert Growth into Tourism Depth",
   direction:"Evaluate tourism performance using overnight stays and average length of stay alongside visitor arrivals, focusing on whether additional arrivals generate additional nights.",
   evidence:"In H1 2026, foreign arrivals increased by 2.04% while foreign overnight stays increased by only 0.74%, and average length of stay declined by approximately 1.27%."
  },
  {
   strategic:"Monitor Leading Indicators",
   direction:"Track monthly arrivals, overnight stays, LOS, source-market performance and forecast error against seasonal benchmarks to identify changes early.",
   evidence:"Rolling out-of-sample testing showed that the seasonal-naive model outperformed the more complex predictive models, with MAPE of about 3.8%, supporting seasonal benchmarks as a practical monitoring baseline."
  }
 ];
 return `<div class="strategy-detail" id="strategyDetail"></div>`;
}

function renderStrategyDetail(index){
 const p=window.strategyPriorities[index];
 const detail=document.getElementById("strategyDetail");
 if(!p || !detail) return;
 detail.innerHTML=`
  <div class="strategy-field">
   <div class="strategy-field-label">Strategic Priority</div>
   <div class="strategy-field-value">${p.strategic}</div>
  </div>
  <div class="strategy-field">
   <div class="strategy-field-label">Recommended Direction</div>
   <div class="strategy-field-value">${p.direction}</div>
  </div>
  <div class="strategy-field">
   <div class="strategy-field-label">Evidence Basis</div>
   <div class="strategy-field-value evidence">${p.evidence}</div>
  </div>
`;
}

function bindStrategyTabs(){
 const bigTabs=[...document.querySelectorAll("#kpis .kpi")];
 if(!bigTabs.length) return;

 const activate=(index)=>{
  bigTabs.forEach((tab,i)=>tab.classList.toggle("strategy-kpi-active",i===index));
  renderStrategyDetail(index);
 };
 activate(0);

 bigTabs.forEach((tab,index)=>{
  tab.onclick=()=>{
   activate(index);
   flash(tab);
  };
 });
}
function fullTimelineChart(){
 const rows=[
   {year:"2021",arrivals:11.66,nights:30.73,los:2.64},
   {year:"2022",arrivals:28.38,nights:67.62,los:2.38},
   {year:"2023",arrivals:34.71,nights:80.38,los:2.32},
   {year:"2024",arrivals:37.42,nights:84.79,los:2.27},
   {year:"2025",arrivals:37.13,nights:83.08,los:2.24},
   {year:"2026 YTD",arrivals:16.35,nights:36.25,los:2.218}
 ];
 const W=760,H=350,padL=60,padR=28,padT=28,padB=52;
 const x=i=>padL+i*((W-padL-padR)/(rows.length-1));
 const y=v=>padT+(90-v)/90*(H-padT-padB);
 const losY=v=>padT+(2.8-v)/(2.8-2.0)*(H-padT-padB);
 const line=(key,fn,cls)=>rows.map((r,i)=>`${i?'L':'M'} ${x(i)} ${fn(r[key])}`).join(" ");
 const marks=(key,fn,unit)=>rows.map((r,i)=>`<circle class="mark" cx="${x(i)}" cy="${fn(r[key])}" r="5" data-tip="${r.year} ${key==="arrivals"?"Foreign arrivals":key==="nights"?"Foreign nights":"Average stay"}: ${r[key]}${unit}"></circle>`).join("");
 const labels=rows.map((r,i)=>`<text x="${x(i)}" y="${H-17}" text-anchor="middle" font-size="11">${r.year}</text>`).join("");
 const grid=[0,30,60,90].map(v=>`<line x1="${padL}" y1="${y(v)}" x2="${W-padR}" y2="${y(v)}" stroke="#dfe5ea"/><text x="${padL-10}" y="${y(v)+4}" text-anchor="end" font-size="10">${v}M</text>`).join("");
 return `<svg viewBox="0 0 ${W} ${H}" role="img" aria-label="Germany international tourism trend from 2021 through 2026 year to date">
   ${grid}
   <path d="${line("nights",y)}" fill="none" stroke="#1565c0" stroke-width="4"/>
   <path d="${line("arrivals",y)}" fill="none" stroke="#d71920" stroke-width="4"/>
   <path d="${line("los",losY)}" fill="none" stroke="#f1b514" stroke-width="3" stroke-dasharray="7 5"/>
   ${marks("nights",y,"M")}
   ${marks("arrivals",y,"M")}
   ${marks("los",losY," nights")}
   ${labels}
   <line x1="${x(5)-24}" y1="${padT}" x2="${x(5)-24}" y2="${H-padB}" stroke="#7b8792" stroke-dasharray="4 4"/>
   <text x="${x(5)}" y="${padT+12}" text-anchor="middle" font-size="10" font-weight="700">YTD Jan–Jun</text>
 </svg>`;
}

function competitiveAnalysisChart(){
 const countries=[
  {name:"Malta",a:1.0,los:4.7,n:4.7,ar:"",nr:"",lr:""},
  {name:"Cyprus",a:1.3,los:5.1,n:6.6,ar:"",nr:"",lr:""},
  {name:"Croatia",a:2.0,los:5.5,n:11.0,ar:"",nr:"",lr:""},
  {name:"Greece",a:3.2,los:5.9,n:18.9,ar:"",nr:"",lr:""},
  {name:"Portugal",a:2.7,los:4.4,n:11.9,ar:"",nr:"",lr:""},
  {name:"Ireland",a:16.5,los:4.9,n:80.9,ar:"",nr:"",lr:""},
  {name:"Denmark",a:18.5,los:3.1,n:57.4,ar:"",nr:"",lr:""},
  {name:"Sweden",a:20.5,los:2.9,n:59.5,ar:"",nr:"",lr:""},
  {name:"Netherlands",a:27.0,los:4.7,n:126.9,ar:"",nr:"",lr:""},
  {name:"Austria",a:28.2,los:3.3,n:93.1,ar:"",nr:"",lr:""},
  {name:"France",a:54.5,los:2.6,n:141.7,ar:"",nr:"",lr:""},
  {name:"Italy",a:74.0,los:3.4,n:251.6,ar:"",nr:"",lr:""},
  {name:"Spain",a:77.8,los:4.1,n:319.0,ar:"",nr:"",lr:""},
  {name:"Germany",a:37.42,los:2.27,n:84.79,ar:"4",nr:"7",lr:"21"}
 ];
 const W=820,H=370,L=64,R=24,T=28,B=54;
 const x=v=>L+(v/82)*(W-L-R);
 const y=v=>T+((6.2-v)/6.2)*(H-T-B);
 const mean=3.17;

 const countryTip=c=>{
   let tip=`${c.name}\n\nForeign arrivals: ${c.a.toFixed(c.name==="Germany"?2:1)}M`;
   if(c.ar) tip+=`\nEU arrivals rank: ${c.ar} of 27`;
   tip+=`\n\nForeign nights: ${c.n.toFixed(2)}M`;
   if(c.nr) tip+=`\nEU nights rank: ${c.nr} of 27`;
   tip+=`\n\nAverage length of stay: ${c.los.toFixed(2)} nights`;
   if(c.lr) tip+=`\nEU LOS rank: ${c.lr} of 27`;
   return tip;
 };

 const dots=countries.map(c=>`
  <circle class="mark" cx="${x(c.a)}" cy="${y(c.los)}" r="${c.name==="Germany"?7:5}"
   fill="${c.name==="Germany"?"#f1b514":"#337caf"}"
   data-tip="${countryTip(c).replace(/\n/g,'&#10;')}"></circle>
  ${c.name==="Germany"?`<text x="${x(c.a)}" y="${y(c.los)+24}" text-anchor="middle" font-size="11" font-weight="700">Germany</text>`:""}
 `).join("");

 const benchmarks=[
  {name:"Germany Actual",los:2.27,illustrative:84.79,v:0,p:0,status:"Actual baseline"},
  {name:"EU27 Median",los:2.49,illustrative:93.18,v:8.39,p:9.89,status:"Illustrative counterfactual"},
  {name:"France",los:2.58,illustrative:96.43,v:11.64,p:13.73,status:"Illustrative counterfactual"},
  {name:"Netherlands",los:2.90,illustrative:108.30,v:23.51,p:27.73,status:"Illustrative counterfactual"},
  {name:"Austria",los:3.30,illustrative:123.43,v:38.64,p:45.58,status:"Illustrative counterfactual"},
  {name:"Weighted EU27",los:3.34,illustrative:124.99,v:40.20,p:47.41,status:"Illustrative counterfactual"}
 ];
 const BW=820,BH=315,BL=132,BR=82,BT=24,BB=36,max=45;
 const bx=v=>BL+(v/max)*(BW-BL-BR);
 const bars=benchmarks.map((d,i)=>{
   const yy=BT+i*31, ww=bx(d.v)-BL;
   const tip=`${d.name}\n\nBenchmark LOS: ${d.los.toFixed(2)} nights\nIllustrative Foreign Nights: ${d.illustrative.toFixed(2)}M\n\nAdditional Nights Vs Actual: ${d.v.toFixed(2)}M\nIncrease vs actual: +${d.p.toFixed(2)}%\n\nScenario type: ${d.status}`;
   return `<text x="${BL-10}" y="${yy+15}" text-anchor="end" font-size="12" fill="#66727c">${d.name}</text>
   <rect class="mark" x="${BL}" y="${yy}" width="${Math.max(2,ww)}" height="20" rx="2"
    fill="${i===0?"#bfc5ca":"#f1c40f"}"
    data-tip="${tip.replace(/\n/g,'&#10;')}"></rect>
   <text x="${Math.max(BL+8,bx(d.v)+7)}" y="${yy+15}" font-size="12">${d.v.toFixed(2)}M | +${d.p.toFixed(2)}%</text>`;
 }).join("");

 return `<div class="dual-chart-grid">
  <section class="dual-chart-panel">
    <div class="dual-chart-title">Germany Combines High Tourism Scale with Below-Average Stay Duration</div>
    <div class="dual-chart-subtitle">2024 foreign arrivals vs. average length of stay across 27 EU countries</div>
    <svg viewBox="0 0 ${W} ${H}" style="width:100%;height:auto">
     <line x1="${L}" y1="${y(mean)}" x2="${W-R}" y2="${y(mean)}" stroke="#8d969e" stroke-width="1.5"/>
     <text x="${L+4}" y="${y(mean)-7}" font-size="11" fill="#6f7880">EU mean LOS: 3.17 nights</text>
     ${[0,20,40,60,80].map(v=>`<line x1="${x(v)}" y1="${T}" x2="${x(v)}" y2="${H-B}" stroke="#edf0f2"/><text x="${x(v)}" y="${H-25}" text-anchor="middle" font-size="10" fill="#7b858d">${v}M</text>`).join("")}
     ${[0,2,4,6].map(v=>`<line x1="${L}" y1="${y(v)}" x2="${W-R}" y2="${y(v)}" stroke="#edf0f2"/><text x="${L-9}" y="${y(v)+4}" text-anchor="end" font-size="10" fill="#7b858d">${v}</text>`).join("")}
     ${dots}
     <text x="${(L+W-R)/2}" y="${H-5}" text-anchor="middle" font-size="12">Foreign Arrivals (2024)</text>
     <text transform="translate(14 ${(T+H-B)/2}) rotate(-90)" text-anchor="middle" font-size="12">Average Length of Stay (Nights)</text>
    </svg>
  </section>

  <section class="dual-chart-panel">
    <div class="dual-chart-title">Longer Stays Could Materially Increase Germany's Overnight Volume</div>
    <div class="dual-chart-subtitle">Illustrative 2024 counterfactuals holding Germany's foreign arrivals constant at 37.42 million</div>
    <svg viewBox="0 0 ${BW} ${BH}" style="width:100%;height:auto">
     ${[0,10,20,30,40].map(v=>`<line x1="${bx(v)}" y1="${BT}" x2="${bx(v)}" y2="${BH-BB}" stroke="#edf0f2"/><text x="${bx(v)}" y="${BH-8}" text-anchor="middle" font-size="10" fill="#7b858d">${v}M</text>`).join("")}
     ${bars}
    </svg>
  </section>
 </div>`;
}

function renderChart(type){
 let html="", legend="";
 if(type==="annual"){ chartWrap.innerHTML=fullTimelineChart(); document.getElementById("legend").innerHTML=`<span><i style="background:#d71920"></i>Foreign arrivals</span><span><i style="background:#1565c0"></i>Foreign nights</span><span><i style="background:#f1b514"></i>Average stay</span><span>2026 is Jan–Jun YTD</span>`; bindMarks(); return;html=annualChart();legend=`<span><i class="dot" style="background:#d71920"></i>Arrivals M</span><span><i class="dot" style="background:#1565c0"></i>Overnight stays M</span><span><i class="dot" style="background:#f1b514"></i>Average stay</span>`}
 if(type==="competition"){html=competitiveAnalysisChart();legend=`<span><i class="dot" style="background:#337caf"></i>EU countries</span><span><i class="dot" style="background:#f1b514"></i>Germany / counterfactual opportunity</span>`}
 if(type==="markets"){html=marketChart()}
 if(type==="ytd"){html=tourism2026Chart();legend=`<span><i class="dot" style="background:#f57c00"></i>Foreign Arrivals / Projected</span><span><i class="dot" style="background:#f1b514"></i>Foreign Nights / Observed</span><span><i class="dot" style="background:#a66f52"></i>Average Length of Stay</span>`}
 if(type==="economy"){html=economyChart();legend=`<span><i class="dot" style="background:#f1b514"></i>Travel Expenditure</span><span><i class="dot" style="background:#f57c00"></i>Travel Receipts</span>`}
 if(type==="closing"){html=closingChart();legend="";setTimeout(bindClosingSlideshow,0)}
 if(type==="statistics"){html=statsChart()}
 if(type==="strategy"){html=strategyChart()}
 chartWrap.innerHTML=html; document.getElementById("legend").innerHTML=legend; bindMarks(); if(type==="strategy") bindStrategyTabs();
}
function bindMarks(){
 document.querySelectorAll(".mark").forEach(m=>{
  const show=e=>{tooltip.textContent=m.dataset.tip||"";tooltip.style.display="block";tooltip.style.left=(e.clientX+14)+"px";tooltip.style.top=(e.clientY+14)+"px"};
  m.addEventListener("mousemove",show);m.addEventListener("mouseenter",show);m.addEventListener("mouseleave",()=>tooltip.style.display="none");
  m.addEventListener("click",e=>{show(e);flash(document.getElementById("graphCard"))});
 });
}



const viewYearOptions={
 performance:["2021","2022","2023","2024","2025","2026 YTD"],
 competition:["2024"],
 markets:["2024"],
 ytd:["2026 YTD"],
 economy:["2021","2022","2023","2024","2025"],
 strategy:["2021","2022","2023","2024","2025","2026 YTD"],
 closing:[]
};

const annualContext={
 "2021":{arrivals:"11.66 M",nights:"30.73 M",los:"2.64 nights",text:"2021 recovery baseline: 11.66M foreign arrivals, 30.73M foreign nights, and 2.64 nights average stay."},
 "2022":{arrivals:"28.38 M",nights:"67.62 M",los:"2.38 nights",text:"2022 rebound: 28.38M foreign arrivals, 67.62M foreign nights, and 2.38 nights average stay."},
 "2023":{arrivals:"34.71 M",nights:"80.38 M",los:"2.32 nights",text:"2023: 34.71M foreign arrivals, 80.38M foreign nights, and 2.32 nights average stay."},
 "2024":{arrivals:"37.42 M",nights:"84.79 M",los:"2.27 nights",text:"2024: 37.42M foreign arrivals, 84.79M foreign nights, 2.27 nights average stay; EU27 ranks #4 arrivals, #7 nights, #21 LOS."},
 "2025":{arrivals:"37.13 M",nights:"83.08 M",los:"2.24 nights",text:"2025: 37.13M foreign arrivals, 83.08M foreign nights, and 2.24 nights average stay."},
 "2026 YTD":{arrivals:"16.35 M",nights:"36.25 M",los:"2.218 nights",text:"H1 2026: 16.35M foreign arrivals and 36.25M foreign nights. Versus H1 2025, arrivals rose 2.04%, nights rose 0.74%, and LOS declined 1.27%."}
};

const economyByYear={
 "2021":{receipts:18827,expenditure:43126,balance:-24300,coverage:43.66,status:"Final"},
 "2022":{receipts:30257,expenditure:85019,balance:-54762,coverage:35.59,status:"Final"},
 "2023":{receipts:34992,expenditure:106642,balance:-71650,coverage:32.81,status:"Final"},
 "2024":{receipts:37055,expenditure:106822,balance:-69767,coverage:34.69,status:"Final"},
 "2025":{receipts:37772,expenditure:114594,balance:-76822,coverage:32.96,status:"Provisional"}
};

function syncYearOptions(key){
 const select=document.getElementById("period");
 if(!select) return;
 const allowed=viewYearOptions[key]||[];
 const prior=select.value;
 select.innerHTML=allowed.map(y=>`<option value="${y}">${y}</option>`).join("");
 if(allowed.includes(prior)) select.value=prior;
 else {
   const defaults={performance:"2024",competition:"2024",markets:"2024",ytd:"2026 YTD",economy:"2024",strategy:"2024"};
   select.value=allowed.includes(defaults[key])?defaults[key]:(allowed[0]||"");
 }
 const hint=document.getElementById("yearScopeHint");
 if(hint){
   const labels={
    performance:"Historical 2021–2025; 2026 YTD",
    competition:"Project EU27 benchmark: 2024",
    markets:"Source-market analysis: 2024",
    ytd:"Current-performance analysis: H1 2026",
    economy:"BOP travel flows: 2021–2025",
    strategy:"Selected-year context; recommendations use combined evidence"
   };
   hint.textContent=labels[key]||"";
 }
 select.title = allowed.length===1
   ? `This project analysis is defined for ${allowed[0]}.`
   : "Choose a year supported by the project dataset/notebooks.";
}
function setView(key){
 document.body.setAttribute("data-view",key);
 const v=views[key];
 syncYearOptions(key);

 const focusSelect=document.getElementById("analysisFocus");
 if(focusSelect) focusSelect.value=key;
 const periodControl=document.getElementById("periodControl");
 if(periodControl) periodControl.classList.toggle("hidden",!["performance","economy"].includes(key));
 const evidenceCard=document.getElementById("evidenceCard");
 if(evidenceCard) evidenceCard.style.display = key==="performance" ? "block" : "none";

 document.getElementById("heroTitle").textContent=v.title;
 document.getElementById("heroSub").textContent=v.sub;
 document.getElementById("graphTitle").textContent=v.graphTitle;
 document.getElementById("graphDesc").textContent=v.graphDesc;
 document.getElementById("finding").textContent=v.finding;
 document.getElementById("implication").textContent=v.implication;
 document.getElementById("transition").textContent=v.transition;
 document.getElementById("sourceNote").textContent=v.source;

 if(key==="strategy"){
  kpis.innerHTML=v.kpis.map((x,i)=>`<button class="kpi" type="button"><div class="value" style="font-size:18px;margin:0">Priority ${i+1}</div></button>`).join("");
 } else {
  kpis.innerHTML=v.kpis.map(x=>`<button class="kpi" type="button"><div class="label">${x[0]}</div><div class="value">${x[1]}</div><div class="note">${x[2]}</div></button>`).join("");
 }
 renderChart(v.chart);
 document.querySelectorAll(".kpi").forEach(k=>k.addEventListener("click",()=>flash(k)));

 applyPeriod();
 if(key!=="performance") setTimeout(()=>{flash(document.getElementById("graphCard"));flash(document.getElementById("photoCard"));},25);
}
photoCard.addEventListener("click",e=>{if(!e.target.closest(".slide-btn")) flash(photoCard);});
document.getElementById("prevPhoto").addEventListener("click",e=>{e.stopPropagation();showTourismPhoto(photoIndex-1,true);});
document.getElementById("nextPhoto").addEventListener("click",e=>{e.stopPropagation();showTourismPhoto(photoIndex+1,true);});
document.getElementById("pausePhoto").addEventListener("click",e=>{e.stopPropagation();togglePhotoSlideshow();});
const periodViews={
 "2021":{
   performance:{
     kpis:[
      ["Foreign Arrivals","11.66 M","Observed 2021"],
      ["Foreign Nights","30.73 M","Observed 2021"],
      ["Avg. Stay","2.64 nights","Highest in displayed series"],
      ["Period","2021","Recovery baseline"]
     ],
     finding:"Germany recorded 11.66 million foreign arrivals and 30.73 million foreign nights in 2021, with an average stay of 2.64 nights.",
     implication:"2021 provides the recovery baseline. Visitor scale subsequently rebounded strongly, while average stay moved in the opposite direction.",
     source:"2021 full-year observed international accommodation data."
   }
 },
 "2022":{
   performance:{
     kpis:[
      ["Foreign Arrivals","28.38 M","Observed 2022"],
      ["Foreign Nights","67.62 M","Observed 2022"],
      ["Avg. Stay","2.38 nights","Down from 2.64 in 2021"],
      ["Period","2022","Full year"]
     ],
     finding:"International tourism rebounded sharply in 2022, reaching 28.38 million arrivals and 67.62 million nights.",
     implication:"The recovery restored visitor volume quickly, but average stay shortened to 2.38 nights.",
     source:"2022 full-year observed international accommodation data."
   }
 },
 "2023":{
   performance:{
     kpis:[
      ["Foreign Arrivals","34.71 M","Observed 2023"],
      ["Foreign Nights","80.38 M","Observed 2023"],
      ["Avg. Stay","2.32 nights","Continued decline"],
      ["Period","2023","Full year"]
     ],
     finding:"By 2023, Germany had reached 34.71 million foreign arrivals and 80.38 million foreign nights.",
     implication:"Scale continued to recover, but average stay fell further to 2.32 nights, reinforcing the tourism-depth question.",
     source:"2023 full-year observed international accommodation data."
   }
 },
 "2024":{
   performance:{
     kpis:[
      ["Foreign Arrivals","37.42 M","4th in EU27"],
      ["Foreign Nights","84.79 M","7th in EU27"],
      ["Avg. Stay","2.27 nights","21st in EU27"],
      ["EU Median LOS","2.49 nights","Germany below benchmark"]
     ],
     finding:"In 2024, Germany combined strong international visitor scale with comparatively short stay duration.",
     implication:"Germany's strongest opportunity is to improve tourism depth, not simply visitor acquisition.",
     source:"2024 full-year observed benchmark · Eurostat accommodation statistics."
   }
 },
 "2025":{
   performance:{
     kpis:[
      ["Foreign Arrivals","37.13 M","Observed 2025"],
      ["Foreign Nights","83.08 M","Observed 2025"],
      ["Avg. Stay","2.24 nights","Below 2024 level"],
      ["Receipts","€37.8 B","2025 provisional"]
     ],
     finding:"In 2025, foreign arrivals and overnight stays were slightly below 2024, while average stay declined further to 2.24 nights.",
     implication:"The persistence of shorter stays strengthens the case for monitoring tourism depth alongside visitor volume.",
     source:"2025 full-year observed accommodation results · travel receipts provisional."
   }
 },
 "2026 YTD":{
   performance:{
     kpis:[
      ["Foreign Arrivals","16.35 M","Jan–Jun 2026"],
      ["Foreign Nights","36.25 M","Jan–Jun 2026"],
      ["Avg. Stay","2.23 nights","YTD arrivals ÷ nights"],
      ["Foreign Nights YoY","+0.5%","H1 2026 vs H1 2025"]
     ],
     finding:"January–June 2026 recorded about 16.35 million international arrivals and 36.25 million international overnight stays. This is a partial-year observation.",
     implication:"2026 YTD must be compared with January–June 2025, not with full-year totals. The dashboard therefore separates the partial-year point visually.",
     source:"2026 YTD = January–June provisional data. Destatis reports 36.4M foreign overnight stays in H1 2026, +0.5% year on year."
   }
 }
};


function updateEvidence(selectedYear){
 const grid=document.getElementById("evidenceGrid");
 if(!grid) return;

 const annualEvidence={
  "2021":[
   ["Observed Scale","11.66M arrivals","30.73M foreign nights"],
   ["Stay Duration","2.64 nights","Highest LOS in the displayed annual series"],
   ["Strategic Context","Recovery baseline","Post-disruption starting point for the trend"]
  ],
  "2022":[
   ["Observed Scale","28.38M arrivals","67.62M foreign nights"],
   ["Stay Duration","2.38 nights","Down from 2.64 nights in 2021"],
   ["Strategic Context","Strong rebound","Visitor scale recovered faster than stay duration"]
  ],
  "2023":[
   ["Observed Scale","34.71M arrivals","80.38M foreign nights"],
   ["Stay Duration","2.32 nights","Average stay continued to decline"],
   ["Strategic Context","Scale strengthened","Tourism depth remained the emerging constraint"]
  ],
  "2024":[
   ["EU Position","#4 arrivals · #7 nights","Strong international visitor scale in the EU27 benchmark"],
   ["Stay Duration","2.27 nights · #21","Below the EU median LOS of 2.49 nights"],
   ["Depth Evidence","≈8.4M nights","Illustrative gap if Germany matched the EU median LOS"]
  ],
  "2025":[
   ["Observed Scale","37.13M arrivals","83.08M foreign nights"],
   ["Stay Duration","2.24 nights","Further decline from 2024"],
   ["Economic Context","€37.8B receipts","2025 travel receipts, provisional"]
  ],
  "2026 YTD":[
   ["H1 Momentum","+2.04% arrivals","Compared with H1 2025"],
   ["Overnight Conversion","+0.74% nights","Nights growth trails arrivals growth"],
   ["Stay Duration","−1.27% LOS","Average stay continues to weaken"]
  ]
 };

 const items=annualEvidence[selectedYear] || annualEvidence["2024"];
 grid.innerHTML=items.map(x=>`
  <div class="evidence-item">
    <div class="e-title">${x[0]}</div>
    <div class="e-value">${x[1]}</div>
    <div class="e-note">${x[2]}</div>
  </div>`).join("");
}
function applyPeriod(){
 const select=document.getElementById("period");
 const selectedYear=select?.value;
 const selectedView=document.getElementById("analysisFocus").value;
 if(selectedView==="closing" || !selectedYear) return;

 if(selectedView==="performance"){
   const p=periodViews[selectedYear].performance;
   updateEvidence(selectedYear);
   kpis.innerHTML=p.kpis.map(x=>`<button class="kpi" type="button"><div class="label">${x[0]}</div><div class="value">${x[1]}</div><div class="note">${x[2]}</div></button>`).join("");
   document.getElementById("finding").textContent=p.finding;
   document.getElementById("implication").textContent=p.implication;
   document.getElementById("sourceNote").textContent=p.source;
   renderChart("annual");

   requestAnimationFrame(()=>{
    document.querySelectorAll(".mark").forEach(m=>{
     const tip=m.dataset.tip||"";
     const match=selectedYear==="2026 YTD" ? tip.startsWith("2026 YTD ") : tip.startsWith(selectedYear+" ");
     if(m.tagName.toLowerCase()==="circle"){
       m.setAttribute("r",match?"9":"5");
       m.style.opacity=match?"1":".42";
       m.style.filter=match?"drop-shadow(0 0 5px rgba(215,25,32,.75))":"none";
     }
    });
   });
 }
 else if(selectedView==="competition"){
   // The final EU27 benchmark in the project is 2024.
   const c=annualContext["2024"];
   kpis.innerHTML=[
    ["EU Arrivals Rank","#4","2024 EU27"],
    ["EU Nights Rank","#7","2024 EU27"],
    ["Average Stay",c.los,"Germany 2024"],
    ["EU LOS Rank","#21","2024 EU27"]
   ].map(x=>`<button class="kpi" type="button"><div class="label">${x[0]}</div><div class="value">${x[1]}</div><div class="note">${x[2]}</div></button>`).join("");
   document.getElementById("finding").textContent="The complete project benchmark is 2024: Germany ranked #4 in foreign arrivals, #7 in foreign nights and #21 in average length of stay.";
   document.getElementById("implication").textContent="Germany's 2.27-night stay was below the EU27 median of 2.49 and mean of 3.17 nights, pointing to a tourism-depth opportunity.";
   document.getElementById("sourceNote").textContent="05_germany_eu_benchmark.ipynb · 2024 EU27 benchmark.";
   renderChart("competition");
 }
 else if(selectedView==="markets"){
   kpis.innerHTML=views.markets.kpis.map(x=>`<button class="kpi" type="button"><div class="label">${x[0]}</div><div class="value">${x[1]}</div><div class="note">${x[2]}</div></button>`).join("");
   document.getElementById("finding").textContent="The source-market strategy is defined from 2024 overnight scale and 2023–2024 growth, with average stay and strategic role.";
   document.getElementById("implication").textContent="The 2024 evidence supports differentiated growth, retention, stay-extension and niche-market strategies rather than one uniform approach.";
   document.getElementById("sourceNote").textContent="06_source_market_analysis.ipynb · 2024 overnight scale and 2023–2024 growth.";
   renderChart("markets");
 }
 else if(selectedView==="ytd"){
   kpis.innerHTML=[
    ["H1 Foreign Arrivals","16.35 M","+2.04% vs H1 2025"],
    ["H1 Foreign Nights","36.25 M","+0.74% vs H1 2025"],
    ["H1 Avg. Stay","2.218 nights","−1.27% vs H1 2025"],
    ["Period","Jan–Jun 2026","Observed YTD"]
   ].map(x=>`<button class="kpi" type="button"><div class="label">${x[0]}</div><div class="value">${x[1]}</div><div class="note">${x[2]}</div></button>`).join("");
   document.getElementById("finding").textContent="H1 2026 arrivals increased by 2.04%, while foreign nights increased by 0.74% and average stay declined by 1.27%.";
   document.getElementById("implication").textContent="Arrival growth is not translating proportionally into overnight-stay growth.";
   document.getElementById("sourceNote").textContent="08_2026_ytd_analysis.ipynb · January–June 2026 versus January–June 2025.";
   renderChart("ytd");
 }
 else if(selectedView==="economy"){
   const e=economyByYear[selectedYear];
   const eur=v=>`€${(v/1000).toFixed(1)} B`;
   kpis.innerHTML=[
    ["Travel Receipts",eur(e.receipts),e.status],
    ["Travel Expenditure",eur(e.expenditure),e.status],
    ["Travel Balance",eur(e.balance),e.status],
    ["Receipts Coverage",`${e.coverage.toFixed(2)}%`,"of expenditure"]
   ].map(x=>`<button class="kpi" type="button"><div class="label">${x[0]}</div><div class="value">${x[1]}</div><div class="note">${x[2]}</div></button>`).join("");
   document.getElementById("finding").textContent=`In ${selectedYear}, travel receipts were ${eur(e.receipts)} and travel expenditure was ${eur(e.expenditure)}, producing a travel balance of ${eur(e.balance)}.`;
   document.getElementById("implication").textContent=`Receipts covered ${e.coverage.toFixed(2)}% of expenditure. This Balance of Payments context is not a measure of tourism-industry profitability.`;
   document.getElementById("sourceNote").textContent=`07_economic_analysis.ipynb · Balance of Payments travel flows · ${selectedYear}${e.status==="Provisional"?" provisional":""}.`;
   renderChart("economy");
 }
 else if(selectedView==="strategy"){
   window.strategyYearContext=annualContext[selectedYear].text;
   document.getElementById("sourceNote").textContent=`10_strategic_analysis.ipynb · selected-year context: ${selectedYear}; recommendations synthesize evidence across the project.`;
   renderChart("strategy");
 }

 document.querySelectorAll(".kpi").forEach(k=>k.addEventListener("click",()=>flash(k)));
 flash(document.getElementById("graphCard"));
}


let dashboardZoom = 1;
let manualZoomOffset = 0;

function setDashboardScale(scale){
  dashboardZoom = Math.max(0.45, Math.min(1.12, scale));
  const app = document.getElementById("app");
  if(!app) return;

  app.style.transform = `scale(${dashboardZoom})`;

  const label = document.getElementById("fitLabel");
  if(label) label.textContent = `${Math.round(dashboardZoom * 100)}%`;
}

function fitDashboardToViewport(){
  const app = document.getElementById("app");
  if(!app) return;

  // Temporarily measure at natural size.
  app.style.transform = "none";

  requestAnimationFrame(()=>{
    const naturalWidth = Math.max(app.scrollWidth, app.offsetWidth);
    const naturalHeight = Math.max(app.scrollHeight, app.offsetHeight);

    const availableWidth = window.innerWidth - 8;
    const availableHeight = window.innerHeight - 8;

    let fitScale = Math.min(
      availableWidth / naturalWidth,
      availableHeight / naturalHeight,
      1
    );

    // Reflow is the primary fit mechanism. Scaling is only a final safeguard
    // when the browser viewport is smaller than the natural dashboard.
    fitScale = fitScale * 0.997;
    setDashboardScale(fitScale + manualZoomOffset);
  });
}

function refitAfterRender(){
  requestAnimationFrame(()=>{
    requestAnimationFrame(fitDashboardToViewport);
  });
  setTimeout(fitDashboardToViewport,120);
  setTimeout(fitDashboardToViewport,350);
}

document.getElementById("zoomOut")?.addEventListener("click",()=>{
  manualZoomOffset -= 0.04;
  fitDashboardToViewport();
});
document.getElementById("zoomIn")?.addEventListener("click",()=>{
  manualZoomOffset += 0.04;
  fitDashboardToViewport();
});
document.getElementById("fitScreen")?.addEventListener("click",()=>{
  manualZoomOffset = 0;
  fitDashboardToViewport();
});
window.addEventListener("resize",()=>{
  manualZoomOffset = 0;
  fitDashboardToViewport();
});

document.getElementById("period").addEventListener("change",()=>{applyPeriod();refitAfterRender();});
document.getElementById("analysisFocus").addEventListener("change",e=>{setView(e.target.value);refitAfterRender();});

setView("performance");
applyPeriod();
showTourismPhoto(0);
restartPhotoTimer();
refitAfterRender();
</script>
</body>
</html>"""

print(f"Dashboard source loaded: {len(DASHBOARD_HTML_SOURCE):,} characters")

In [ ]:
# Save a documented copy of the current dashboard into the project presentation folder

dashboard_output = (
    presentation_dir
    / "Germany_Tourism_Dashboard_V36_3_Targeted_Changes.html"
)

dashboard_output.write_text(
    DASHBOARD_HTML_SOURCE,
    encoding="utf-8"
)

print("Dashboard copy saved:", dashboard_output)


# 16. What to answer during the presentation

### “Did you code the dashboard?”

A precise answer:

> Yes. The analytical pipeline was developed in Jupyter notebooks using Python, with SQL and Tableau supporting the project. For the final presentation interface, I used HTML, CSS, JavaScript and SVG. I also used AI as a coding and design assistant during iterative front-end development, but the dashboard is grounded in the outputs and logic of my own project notebooks. I validated the metrics against the processed files before using them in the interface.

### “Can you show where a dashboard number comes from?”

Example:

> The H1 2026 arrival growth shown in the dashboard comes from Notebook 08. That notebook aggregates January–June arrivals for 2025 and 2026 and calculates the year-on-year percentage change using `pct_change()`. The result is saved to `germany_2026_ytd_summary.csv`, which this dashboard notebook loads and validates.

### “Where does the EU rank come from?”

> Notebook 05 merges the EU27 foreign arrivals and overnight-stay tables, calculates average length of stay as nights divided by arrivals, ranks the countries, and saves the 2024 benchmark. The dashboard reads that output. Germany is fourth in arrivals, seventh in nights and twenty-first in average stay.

### “Where do the recommendations come from?”

> Notebook 10 integrates the benchmark, source-market, economic, YTD and model evidence. The dashboard recommendation cards are therefore based on the strategic output of Notebook 10 rather than being written independently for the presentation.

### “Why not calculate everything directly inside the HTML?”

> I separated analysis from presentation. Python and the notebooks are the reproducible analytical layer. The HTML dashboard is the interactive presentation layer. This reduces the risk of changing analytical logic inside a visualization file and makes the data lineage easier to audit.


# 17. 30-second technical explanation

> The dashboard is the final presentation layer of my existing analytics pipeline. Notebooks 05 through 10 produce the analytical outputs, and Notebook 11 prepares the presentation tables. This notebook reads those outputs, performs KPI validation, converts the results to JSON/JavaScript-ready objects, and documents the HTML/CSS/JavaScript front end. So if a value changes upstream, the dashboard data layer can be regenerated from the project files rather than manually recalculated in the presentation.
